In [ ]:


# -*- coding: utf-8 -*-
# ## Weighting refactor for NS HLSI
#
# This script separates the local score signal (``init``: HLSI vs CE-HLSI)
# from the global SNIS weighting (``init_weights``: ``L``, ``WC``, ``PoU``).
# The default comparison below is the requested 7-way suite: HLSI / CE-HLSI
# with L, WC, and PoU weighting, plus prior-initialized MALA as reference.
"""NS_ce_hlsi.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1HSuKyx-GivU9eALaV8qyREMoyzI2prQ9
"""

import os
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

# ==========================================
# CONFIGURATION GENERATOR
# ==========================================
# Use this cell to generate specific sensor configurations before running the solver.

# 1. Parameters
N = 32
num_observation = 100  # <--- Change this to generate different sensor counts (e.g., 10, 50)
num_truncated_series = 32 # Latent dimension for NS (Standard is 24)
seed = 42 # Change seed for different random sensor placements

print(f"Generating configuration for {num_observation} observations...")

# 2. Generate Random Observation Locations
key = jax.random.PRNGKey(seed)
# Total grid points = N*N. We select 'num_observation' unique indices.
# Note: We exclude the last point N*N-1 if strictly following original script logic,
# but generally choice over N*N is fine. Original script used arange(N*N-1).
possible_indices = jnp.arange(N * N)
obs_indices = jax.random.choice(key, possible_indices, shape=(num_observation,), replace=False)
obs_indices = np.array(obs_indices) # Convert to numpy for saving

# 3. Load and Truncate Basis
# We need the source file 'data/Basis_Modes.csv' which contains the top 100 modes.
try:
    df_modes = pd.read_csv('data/Basis_Modes.csv', header=None)

    # Handle potential header in source file
    if isinstance(df_modes.iloc[0,0], str):
        df_modes = pd.read_csv('data/Basis_Modes.csv')

    modes_raw = df_modes.to_numpy().flatten()

    # The source file usually has 100 modes for 32x32 grid
    # Shape should be (1024, 100) -> size 102400
    # Or (1024, 100) flattened.

    # Reshape logic based on grid size N=32 (1024 points)
    full_dim = N * N
    num_modes_available = modes_raw.size // full_dim

    full_basis = modes_raw.reshape((full_dim, num_modes_available))

    # Truncate to the number of modes we want for the inverse problem
    basis_truncated = full_basis[:, :num_truncated_series]

    print(f"Loaded source basis with {num_modes_available} modes.")
    print(f"Truncated to {num_truncated_series} modes.")

    # 4. Save Files
    # Save Basis.csv
    pd.DataFrame(basis_truncated).to_csv('data/Basis.csv', index=False, header=False)
    print("Saved 'data/Basis.csv'")

    # Save obs_locations.csv
    pd.DataFrame(obs_indices).to_csv('data/obs_locations.csv', index=False, header=False)
    print("Saved 'data/obs_locations.csv'")

except FileNotFoundError:
    print("ERROR: Could not find 'data/Basis_Modes.csv'. This source file is required to generate the truncated basis.")
    # Create dummy files ONLY if you just want to test the pipeline structure without physics accuracy
    # print("Generating DUMMY basis for pipeline testing...")
    # dummy_basis = np.random.randn(N*N, num_truncated_series)
    # pd.DataFrame(dummy_basis).to_csv('data/Basis.csv', index=False, header=False)
    # pd.DataFrame(obs_indices).to_csv('data/obs_locations.csv', index=False, header=False)

print("Configuration generation complete.\n")

import os
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from functools import partial

# Hardware setup
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Uncomment if managing specific GPUs
jax.config.update("jax_enable_x64", True)

print("Initializing JAX Navier-Stokes Physics Engine...")

# ==========================================
# 1. LOAD PHYSICAL CONSTANTS & BASIS
# ==========================================

# Physical model dimensions (Navier Stokes 2D)
N = 32  # Grid size 32x32
dimension_of_PoI = N**2  # 1024
num_observation = 100
num_truncated_series = 24 # Latent dimension (KL modes) we want to use

# Load Basis and Observation Locations
try:
    # Read files with header=None to capture everything, then clean up
    df_Basis = pd.read_csv('data/Basis.csv', header=None)
    df_obs = pd.read_csv('data/obs_locations.csv', header=None)

    # --- Fix for Basis Shape (Handle Off-by-One Header) ---
    basis_raw = df_Basis.to_numpy().flatten()

    # Check if we have exactly 1 extra element (e.g. 102401 instead of 102400)
    if basis_raw.size % dimension_of_PoI == 1:
        print(f"Basis size is {basis_raw.size} (100 x 1024 + 1). Detecting header artifact. Dropping first element.")
        basis_raw = basis_raw[1:]

    # Ensure numeric type (in case the header made the array object-type)
    basis_raw = basis_raw.astype(np.float64)

    total_elements = basis_raw.size

    # Expected columns if we have 'dimension_of_PoI' rows
    if total_elements % dimension_of_PoI != 0:
        raise ValueError(f"Basis file size {total_elements} is not divisible by grid size {dimension_of_PoI} (32x32).")

    num_modes_in_file = total_elements // dimension_of_PoI

    # Reshape to (1024, num_modes_in_file)
    full_basis = np.reshape(basis_raw, (dimension_of_PoI, num_modes_in_file))

    print(f"Basis file found with {num_modes_in_file} modes. Truncating to first {num_truncated_series}...")

    # Truncate to the 24 modes we need
    Basis = jnp.array(full_basis[:, :num_truncated_series])

    # --- Fix for Obs Locations ---
    obs_raw = df_obs.to_numpy().flatten()

    # Handle potential off-by-one in obs file too (just in case)
    if obs_raw.size == num_observation + 1:
         print("Obs file has 1 extra element. Dropping first element.")
         obs_raw = obs_raw[1:]

    obs_raw = obs_raw.astype(int) # Ensure integer for indexing

    # If file has more observations than needed (e.g. 20 vs 100), truncate
    if obs_raw.size > num_observation:
        print(f"Obs file has {obs_raw.size} locations. Using first {num_observation}.")
        obs_raw = obs_raw[:num_observation]
    elif obs_raw.size < num_observation:
        raise ValueError(f"Obs file only has {obs_raw.size} locations, need {num_observation}.")

    obs_locations = jnp.array(obs_raw, dtype=int)

    print(f"Loaded Final Basis shape: {Basis.shape}")
    print(f"Loaded Final Obs Indices shape: {obs_locations.shape}")

except FileNotFoundError:
    raise FileNotFoundError("Please ensure 'data/Basis.csv' and 'data/obs_locations.csv' exist in the directory.")
except Exception as e:
    print(f"Error loading data: {e}")
    raise

# ==========================================
# 2. DEFINE SPECTRAL FORWARD SOLVER (NS)
# ==========================================

# --- NS Solver Constants ---
dx = 1 / N
x = jnp.linspace(0, 1, N)
X, Y = jnp.meshgrid(x, x)

visc = 1e-3
T_end = 10.0
delta_t = 1e-2 * 5 # dt = 0.05
k_max = jnp.floor(N/2.0)
steps = int(jnp.ceil(T_end/delta_t))

# Forcing function: f(x) = 0.1 * (sin(2pi(x+y)) + cos(2pi(x+y)))
f = 0.1 * (jnp.sin(2*jnp.pi*(X + Y)) + jnp.cos(2*jnp.pi*(X + Y)))
f_h = jnp.fft.fft2(f)

# Wavenumbers for FFT derivatives
k = jnp.concatenate((jnp.arange(0, k_max, 1), jnp.arange(-k_max, 0, 1)))
k_y = jnp.tile(k, (N, 1))
k_x = k_y.T

# Negative Laplacian in Fourier space: 4*pi^2*(kx^2 + ky^2)
lap = 4 * (jnp.pi**2) * (k_x**2 + k_y**2)
lap = lap.at[0, 0].set(1.0) # Avoid division by zero at mode (0,0)

# Dealiasing mask (2/3 rule for spectral stability)
dealias = jnp.logical_and(jnp.abs(k_y) < 2./3.*k_max, jnp.abs(k_x) < 2./3.*k_max) * 1.0

@jax.jit
def body_loop(i, w_h):
    """
    Single time-step of the 2D Navier-Stokes solver in Fourier space.
    Solves for vorticity w_h using Crank-Nicolson update.
    """
    # 1. Stream function: solve Poisson equation psi_h = w_h / lap
    psi_h = w_h / lap

    # 2. Compute Velocity field in Fourier space (derivatives of stream function)
    # v_x = d(psi)/dy = 2*pi*i*ky * psi
    # v_y = -d(psi)/dx = -2*pi*i*kx * psi
    q_h = 2 * jnp.pi * k_y * psi_h * 1j
    v_h = -2 * jnp.pi * k_x * psi_h * 1j

    # 3. Compute Vorticity gradients in Fourier space
    w_x_h = 2 * jnp.pi * k_x * w_h * 1j
    w_y_h = 2 * jnp.pi * k_y * w_h * 1j

    # 4. Inverse FFT to physical space to compute non-linear convection
    q = jnp.real(jnp.fft.ifft2(q_h))
    v = jnp.real(jnp.fft.ifft2(v_h))
    w_x = jnp.real(jnp.fft.ifft2(w_x_h))
    w_y = jnp.real(jnp.fft.ifft2(w_y_h))

    # 5. Non-linear term (u . grad) w computed in physical space
    # Transform back to Fourier
    F_h = jnp.fft.fft2(q * w_x + v * w_y)

    # 6. Apply Dealiasing
    F_h = dealias * F_h

    # 7. Crank-Nicolson time integration
    # (1 + 0.5*dt*visc*lap) w_new = (1 - 0.5*dt*visc*lap) w_old - dt*F_h + dt*f_h
    denom = 1.0 + 0.5 * delta_t * visc * lap
    numer = (1.0 - 0.5 * delta_t * visc * lap) * w_h - delta_t * F_h + delta_t * f_h

    w_h_new = numer / denom
    return w_h_new

@jax.jit
def solve_forward(x):
    """
    The Forward Map F(alpha) for Navier-Stokes.
    Input: alpha coefficients (latent space, shape 24)
    Output: Observation vector y_pred (shape 20)
    """
    # 1. Map latent coefficients to Initial Vorticity field on grid
    # Basis shape: (1024, 24), x shape: (24,)
    w0 = jnp.reshape((Basis @ x.T), (N, N))

    # 2. Move to Fourier space
    w_h = jnp.fft.fft2(w0)

    # 3. Time stepping loop (0 to T_end)
    w_h_final = jax.lax.fori_loop(0, steps, body_loop, w_h)

    # 4. Inverse FFT to get final vorticity in physical space
    w_final = jnp.real(jnp.fft.ifft2(w_h_final))

    # 5. Extract Observations at sensor locations
    y_obs = (w_final.flatten())[obs_locations]

    return y_obs

print("Physics Engine Loaded Successfully.")
print(f"Forward map 'solve_forward' is ready. Input dim: {num_truncated_series}, Output dim: {num_observation}")

import torch
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pprint import pformat
import platform
import shutil
import types
import time
import math

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_RESULTS_ROOT = 'run_results'
RUN_RESULTS_DIR = os.path.join(RUN_RESULTS_ROOT, f'navier_stokes_{RUN_TIMESTAMP}')
os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
RUN_RESULTS_STEM = f'navier_stokes_{RUN_TIMESTAMP}'

_PLOT_SAVE_COUNTER = 0
_ORIGINAL_PLT_SHOW = plt.show


def _sanitize_run_results_name(text, max_len=96):
    text = str(text).strip().replace('\n', ' ')
    safe = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in text)
    while '__' in safe:
        safe = safe.replace('__', '_')
    safe = safe.strip('_')
    if not safe:
        safe = 'figure'
    return safe[:max_len]


def _infer_figure_basename(fig, fallback):
    title_candidates = []
    suptitle = getattr(fig, '_suptitle', None)
    if suptitle is not None:
        try:
            txt = suptitle.get_text().strip()
            if txt:
                title_candidates.append(txt)
        except Exception:
            pass
    for ax in fig.axes:
        try:
            txt = ax.get_title().strip()
            if txt:
                title_candidates.append(txt)
                break
        except Exception:
            pass
    if title_candidates:
        return _sanitize_run_results_name(title_candidates[0])
    return _sanitize_run_results_name(fallback)


def _save_all_open_figures_to_run_results():
    global _PLOT_SAVE_COUNTER
    for fig_num in plt.get_fignums():
        fig = plt.figure(fig_num)
        if getattr(fig, '_run_results_saved', False):
            continue
        _PLOT_SAVE_COUNTER += 1
        basename = _infer_figure_basename(fig, f'figure_{_PLOT_SAVE_COUNTER:02d}')
        png_path = os.path.join(
            RUN_RESULTS_DIR,
            f'{RUN_RESULTS_STEM}_figure_{_PLOT_SAVE_COUNTER:02d}_{basename}.png',
        )
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
        fig._run_results_saved = True
        print(f'Saved figure to {png_path}')


def _patched_show(*args, **kwargs):
    _save_all_open_figures_to_run_results()
    return _ORIGINAL_PLT_SHOW(*args, **kwargs)


plt.show = _patched_show


def _summarize_for_repro(value):
    if isinstance(value, (bool, int, float, str, type(None))):
        return repr(value)
    if isinstance(value, np.ndarray):
        if value.size <= 32:
            return repr(value.tolist())
        return (
            f'np.ndarray(shape={value.shape}, dtype={value.dtype}, '
            f'min={float(np.min(value)):.6g}, max={float(np.max(value)):.6g}, '
            f'mean={float(np.mean(value)):.6g}, std={float(np.std(value)):.6g})'
        )
    if torch.is_tensor(value):
        value_cpu = value.detach().cpu()
        if value_cpu.numel() <= 32:
            return repr(value_cpu.tolist())
        return (
            f'torch.Tensor(shape={tuple(value_cpu.shape)}, dtype={value_cpu.dtype}, '
            f'min={float(value_cpu.min().item()):.6g}, max={float(value_cpu.max().item()):.6g}, '
            f'mean={float(value_cpu.double().mean().item()):.6g}, '
            f'std={float(value_cpu.double().std(unbiased=False).item()):.6g})'
        )
    if isinstance(value, (list, tuple, dict, set)):
        try:
            formatted = pformat(value, width=100, compact=False, sort_dicts=False)
        except TypeError:
            formatted = pformat(value, width=100, compact=False)
        if len(formatted) > 5000:
            formatted = formatted[:5000] + '\n... [truncated]'
        return formatted
    if hasattr(value, 'shape') and hasattr(value, 'dtype'):
        shape = getattr(value, 'shape', None)
        dtype = getattr(value, 'dtype', None)
        return f'{type(value).__name__}(shape={shape}, dtype={dtype})'
    return repr(value)


def save_reproducibility_log(extra_sections=None):
    log_path = os.path.join(RUN_RESULTS_DIR, f'{RUN_RESULTS_STEM}_parameters.txt')
    lines = []
    lines.append('Navier-Stokes inversion HLSI run reproducibility log')
    lines.append('=' * 72)
    lines.append(f'run_timestamp = {RUN_TIMESTAMP}')
    lines.append(f'script_path = {globals().get("__file__", "<interactive>")}')
    lines.append(f'python_version = {platform.python_version()}')
    lines.append(f'platform = {platform.platform()}')
    lines.append(f'numpy_version = {np.__version__}')
    lines.append(f'pandas_version = {pd.__version__}')
    lines.append(f'torch_version = {torch.__version__}')
    lines.append(f'jax_version = {jax.__version__}')
    lines.append(f'device = {device}')
    lines.append(f'cuda_available = {torch.cuda.is_available()}')
    lines.append(f'run_results_dir = {RUN_RESULTS_DIR}')
    lines.append('')
    lines.append('Key configuration values')
    lines.append('-' * 72)

    excluded_names = {
        'PRECOMP_NS', 'samples_ns', 'ess_logs_ns', 'metrics_ns', 'results_df', 'results_runinfo_df',
        'mean_fields', 'anchor_vis_fields', 'fields', 'ref_hist', 'y_clean_np'
    }
    config_names = []
    for name, value in globals().items():
        if name in excluded_names:
            continue
        if name.startswith('_'):
            continue
        if isinstance(value, types.ModuleType):
            continue
        if callable(value):
            continue
        if name.isupper() or name in {
            'seed', 'device', 'N_REF_NS', 'SAMPLER_CONFIGS_NS', 'sampler_run_info_ns',
            'display_names_ns', 'reference_key_ns', 'reference_title_ns',
            'num_observation', 'num_truncated_series', 'dimension_of_PoI', 'num_modes_available',
            'obs_indices', 'obs_locations', 'd_lat', 'steps', 'delta_t', 'T_end'
        }:
            config_names.append(name)

    for name in sorted(set(config_names)):
        try:
            lines.append(f'{name} = {_summarize_for_repro(globals()[name])}')
        except Exception as exc:
            lines.append(f'{name} = <unable to serialize: {exc}>')
        lines.append('')

    if extra_sections:
        for section_name, section_value in extra_sections.items():
            lines.append(section_name)
            lines.append('-' * 72)
            lines.append(_summarize_for_repro(section_value))
            lines.append('')

    with open(log_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print(f'Saved reproducibility log to {log_path}')
    return log_path


def zip_run_results_dir():
    _save_all_open_figures_to_run_results()
    zip_path = shutil.make_archive(
        RUN_RESULTS_DIR,
        'zip',
        root_dir=RUN_RESULTS_ROOT,
        base_dir=os.path.basename(RUN_RESULTS_DIR),
    )
    print(f'Compressed run-results directory to {zip_path}')
    return zip_path

# ==========================================
# CONFIGURATION
# ==========================================
jax.config.update("jax_enable_x64", True)
torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Global Dtype set to: {torch.get_default_dtype()}")
print(f"Device: {device}")

# Configuration for Navier-Stokes (adapted from TAEN section 3.2)
ACTIVE_DIM = 24 # Latent dimension (KL modes)
NOISE_STD = 0.025 # 15% Noise level

# Hessian spectral band for HLSI (Section 5.3: Spectral band truncation)
# Eigenvalues of the local precision P_i outside [HESS_MIN, HESS_MAX]
# are discarded and those directions fall back to the Tweedie estimator.
HESS_MIN = 1e-4 # Lower bound: eigenvalues below this are too sloppy
HESS_MAX = 1e6  #1e6    # Upper bound: eigenvalues above this are too stiff


# ==========================================
# 1. BRIDGE: JAX PHYSICS -> PYTORCH SAMPLER
# ==========================================

# A. Define JAX Likelihood Functions (Wrapped)
# solve_forward is assumed to be in context from the previous cell (Spectral NS)
# A. Define JAX Likelihood Functions (Wrapped)
# solve_forward is assumed to be in context from the previous cell (Spectral NS)

@jax.jit
def ns_log_likelihood_jax(alpha_k, y_obs_jax, sigma):
    y_pred = solve_forward(alpha_k)
    resid = y_pred - y_obs_jax
    return -jnp.sum(resid**2) / (2.0 * sigma**2)

# Gradient
ns_grad_lik_jax = jax.jit(jax.grad(ns_log_likelihood_jax, argnums=0))

# --- Hessian options ---
USE_GAUSS_NEWTON_HESSIAN = True  # recommended
# If False, uses exact jax.hessian (often much slower / more fragile through PDE solvers)

# Jacobian of forward map (output_dim x dim); since dim is small (e.g., 15), jacfwd is usually best.
solve_forward_jac_jax = jax.jit(jax.jacfwd(solve_forward))

@jax.jit
def ns_hess_lik_gn_jax(alpha_k, y_obs_jax, sigma):
    # Gauss–Newton Hessian of log-likelihood:
    # log L = -||F(x)-y||^2/(2 sigma^2) => Hess(log L) ≈ -(J^T J)/sigma^2
    J = solve_forward_jac_jax(alpha_k)  # shape: (m, d)
    return -(J.T @ J) / (sigma**2)

ns_hess_lik_exact_jax = jax.jit(jax.hessian(ns_log_likelihood_jax, argnums=0))

def ns_hess_lik_jax(alpha_k, y_obs_jax, sigma):
    return ns_hess_lik_gn_jax(alpha_k, y_obs_jax, sigma) if USE_GAUSS_NEWTON_HESSIAN else ns_hess_lik_exact_jax(alpha_k, y_obs_jax, sigma)

# Vectorize for batch processing
batch_log_lik  = jax.vmap(ns_log_likelihood_jax, in_axes=(0, None, None))
batch_grad_lik = jax.vmap(ns_grad_lik_jax,        in_axes=(0, None, None))
batch_hess_lik = jax.vmap(ns_hess_lik_jax,        in_axes=(0, None, None))

# B. PyTorch Interface Class
class NSLikelihood:
    def __init__(self, y_obs_np, sigma):
        self.y_obs_jax = jnp.array(y_obs_np)
        self.sigma = float(sigma)

    def log_likelihood(self, x_torch):
        x_np = x_torch.detach().cpu().numpy()
        ll_jax = batch_log_lik(x_np, self.y_obs_jax, self.sigma)
        return torch.tensor(np.array(ll_jax), device=x_torch.device, dtype=torch.float64)

    def grad_log_likelihood(self, x_torch):
        x_np = x_torch.detach().cpu().numpy()
        grad_jax = batch_grad_lik(x_np, self.y_obs_jax, self.sigma)
        return torch.tensor(np.array(grad_jax), device=x_torch.device, dtype=torch.float64)

    def hess_log_likelihood(self, x_torch):
        """
        Returns Hessian of log-likelihood at x: shape (N, d, d)
        If USE_GAUSS_NEWTON_HESSIAN=True, this is the GN approximation (recommended).
        """
        x_np = x_torch.detach().cpu().numpy()
        hess_jax = batch_hess_lik(x_np, self.y_obs_jax, self.sigma)
        return torch.tensor(np.array(hess_jax), device=x_torch.device, dtype=torch.float64)

# ==========================================
# 2. PRIOR
# ==========================================
class GaussianPrior(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def sample(self, n):
        return torch.randn(n, self.dim, device=device)

    def log_prob(self, x):
        # log N(0, I)
        return -0.5 * torch.sum(x**2, dim=1) - (self.dim/2.0) * math.log(2*math.pi)

    def score0(self, x):
        # Score of N(0, I) is -x
        return -x

# ==========================================
# 3. SAMPLERS
# ==========================================
def get_posterior_snis_weights(y, t, X_ref, log_lik_ref):
    et = math.exp(-t)
    var_t = 1.0 - math.exp(-2*t)
    mus = et * X_ref
    diff = y.unsqueeze(1) - mus.unsqueeze(0)
    dists_sq = torch.sum(diff**2, dim=2)
    log_kernel = -dists_sq / (2 * var_t)
    log_unnorm = log_kernel + log_lik_ref.unsqueeze(0)
    log_norm = torch.logsumexp(log_unnorm, dim=1, keepdim=True)
    return torch.exp(log_unnorm - log_norm)


def eval_blend_posterior_score(y, t, X_ref, s0_post_ref, log_lik_ref):
    if t < 1e-4: t = 1e-4
    et = math.exp(-t)
    var_t = 1.0 - math.exp(-2*t)
    inv_v = 1.0 / var_t
    eps = 1e-12

    scale_factor = 1.0 / et

    w = get_posterior_snis_weights(y, t, X_ref, log_lik_ref)

    s_kss = scale_factor * torch.einsum('mn,nd->md', w, s0_post_ref)
    mu_x = torch.einsum('mn,nd->md', w, X_ref)
    s_twd = -inv_v * (y - et * mu_x)

    w2 = w**2
    S0 = torch.sum(w2, dim=1)
    den_sn = torch.clamp(1.0 - S0, min=eps)

    a_i = scale_factor * s0_post_ref
    a_norm2 = torch.sum(a_i**2, dim=1)
    S1a = torch.mv(w2, a_norm2)
    S2a = torch.mm(w2, a_i)
    mu_a = s_kss
    mu_a_norm2 = torch.sum(mu_a**2, dim=1)
    num_Vk = S1a - 2.0 * torch.sum(mu_a * S2a, dim=1) + mu_a_norm2 * S0
    Vk = num_Vk / den_sn

    w2_z = torch.mm(w2, X_ref)
    S2b = -inv_v * (y * S0.unsqueeze(1) - et * w2_z)
    y_norm2 = torch.sum(y**2, dim=1)
    z_norm2 = torch.sum(X_ref**2, dim=1)
    term1 = y_norm2 * S0
    term2 = -2.0 * et * torch.sum(y * w2_z, dim=1)
    term3 = (et**2) * torch.mv(w2, z_norm2)
    S1b = (inv_v**2) * (term1 + term2 + term3)
    mu_b = s_twd
    mu_b_norm2 = torch.sum(mu_b**2, dim=1)
    num_Vt = S1b - 2.0 * torch.sum(mu_b * S2b, dim=1) + mu_b_norm2 * S0
    Vt = num_Vt / den_sn

    a_dot_z = torch.sum(a_i * X_ref, dim=1)
    Wa = S2a
    term_c1 = torch.sum(Wa * y, dim=1)
    term_c2 = torch.mv(w2, a_dot_z)
    Sab = -inv_v * (term_c1 - et * term_c2)
    num_C = Sab - torch.sum(mu_a * S2b, dim=1) - torch.sum(mu_b * S2a, dim=1) + torch.sum(mu_a * mu_b, dim=1) * S0
    C = num_C / den_sn

    denom = torch.clamp(Vk + Vt - 2.0 * C, min=eps)
    lam = (Vk - C) / denom
    lam = torch.clamp(lam, 0.0, 0.95)

    return lam.unsqueeze(1) * s_twd + (1.0 - lam.unsqueeze(1)) * s_kss

def get_score_wrapper(y, t, mode, X_ref, s0_post_ref, log_lik_ref):
    t_val = t.item() if isinstance(t, torch.Tensor) else t
    if mode == 'tweedie':
        if t_val < 1e-4: t_val = 1e-4
        et = math.exp(-t_val)
        inv_v = 1.0 / (1 - math.exp(-2*t_val))
        w = get_posterior_snis_weights(y, t_val, X_ref, log_lik_ref)
        mu_x = torch.einsum('mn,nd->md', w, X_ref)
        score = -inv_v * (y - et * mu_x)
    elif mode == 'blend_posterior':
        score = eval_blend_posterior_score(y, t_val, X_ref, s0_post_ref, log_lik_ref)
    return score



# ==========================================
# 3. BATCHED SCORE COMPUTATION (NEW)
# ==========================================

def eval_score_batched(y, t, X_ref_cpu, s0_ref_cpu, log_lik_ref_cpu, batch_size=4096, mode='blend_posterior'):
    """
    Computes Tweedie or Blend Posterior score using streaming batches to avoid OOM.
    Reference data (X_ref_cpu, etc.) stays on CPU.
    """
    t_val = t.item() if isinstance(t, torch.Tensor) else t
    if t_val < 1e-4: t_val = 1e-4

    et = math.exp(-t_val)
    var_t = 1.0 - math.exp(-2 * t_val)
    inv_v = 1.0 / var_t

    # Constants for Blend
    scale_factor = 1.0 / et

    M_query = y.shape[0]
    N_ref = X_ref_cpu.shape[0]

    # --- PASS 1: Find Global Max Log-Weight (Stability) ---
    max_log_w = torch.full((M_query,), -float('inf'), device=y.device)

    for i in range(0, N_ref, batch_size):
        # Move chunk to GPU
        X_batch = X_ref_cpu[i:i+batch_size].to(y.device)
        ll_batch = log_lik_ref_cpu[i:i+batch_size].to(y.device)

        # log_weight = -||y - exp(-t)x||^2 / 2var + log_lik
        mus = et * X_batch
        dists_sq = torch.sum((y.unsqueeze(1) - mus.unsqueeze(0))**2, dim=2)
        log_w_batch = -dists_sq / (2 * var_t) + ll_batch.unsqueeze(0)

        # Update running max
        current_max, _ = torch.max(log_w_batch, dim=1)
        max_log_w = torch.max(max_log_w, current_max)

    # --- PASS 2: Accumulate Moments ---
    # Common Accumulators
    denom_Z = torch.zeros((M_query, 1), device=y.device) # sum(w_tilde)
    numer_mu_x = torch.zeros_like(y)                     # sum(w_tilde * x)

    # Blend Accumulators (Sum of w_tilde^2 * term)
    if mode == 'blend_posterior':
        numer_s0 = torch.zeros_like(y)                   # sum(w_tilde * s0)
        acc_w2 = torch.zeros((M_query, 1), device=y.device)
        acc_w2_s0_norm = torch.zeros((M_query,), device=y.device)
        acc_w2_s0 = torch.zeros_like(y)
        acc_w2_x = torch.zeros_like(y)
        acc_w2_x_norm = torch.zeros((M_query,), device=y.device)
        acc_w2_dot = torch.zeros((M_query,), device=y.device) # sum(w_tilde^2 * (s0 . x))

    for i in range(0, N_ref, batch_size):
        X_batch = X_ref_cpu[i:i+batch_size].to(y.device)
        ll_batch = log_lik_ref_cpu[i:i+batch_size].to(y.device)

        mus = et * X_batch
        dists_sq = torch.sum((y.unsqueeze(1) - mus.unsqueeze(0))**2, dim=2)
        log_w = -dists_sq / (2 * var_t) + ll_batch.unsqueeze(0)

        # Stable Unnormalized Weights: w_tilde = exp(log_w - M)
        w_batch = torch.exp(log_w - max_log_w.unsqueeze(1))

        # 1st Moment Accumulation
        denom_Z += torch.sum(w_batch, dim=1, keepdim=True)
        numer_mu_x += torch.einsum('mb,bd->md', w_batch, X_batch)

        if mode == 'blend_posterior':
            s0_batch = s0_ref_cpu[i:i+batch_size].to(y.device)
            numer_s0 += torch.einsum('mb,bd->md', w_batch, s0_batch)

            # 2nd Moment Accumulation (Squared Weights)
            w2_batch = w_batch**2
            acc_w2 += torch.sum(w2_batch, dim=1, keepdim=True)

            # Precompute batch norms/dots
            s0_sq_batch = torch.sum(s0_batch**2, dim=1)
            x_sq_batch = torch.sum(X_batch**2, dim=1)
            dot_batch = torch.sum(s0_batch * X_batch, dim=1)

            acc_w2_s0_norm += torch.mv(w2_batch, s0_sq_batch)
            acc_w2_s0 += torch.mm(w2_batch, s0_batch)
            acc_w2_x += torch.mm(w2_batch, X_batch)
            acc_w2_x_norm += torch.mv(w2_batch, x_sq_batch)
            acc_w2_dot += torch.mv(w2_batch, dot_batch)

    # --- FINALIZE / RECONSTRUCTION ---
    eps = 1e-12
    # Tweedie Score: -(y - et * E[x])/var_t
    mu_x = numer_mu_x / denom_Z
    score_twd = -inv_v * (y - et * mu_x)

    if mode == 'tweedie':
        return score_twd

    # Blend Logic Reconstruction
    # Key Identity: sum(w_norm^2 * term) = (1/Z^2) * sum(w_tilde^2 * term)
    Z_sq = denom_Z**2

    # 1. Reconstruct Expectations
    mu_a = scale_factor * (numer_s0 / denom_Z)
    mu_b = score_twd # Tweedie is mu_b

    S0 = acc_w2 / Z_sq

    # S1a = sum(w^2 ||a||^2) -> a = scale * s0 -> scale^2 * sum(w^2 ||s0||^2)
    S1a = (scale_factor**2) * (acc_w2_s0_norm.unsqueeze(1) / Z_sq)

    # S2a = sum(w^2 a) -> scale * sum(w^2 s0)
    S2a = scale_factor * (acc_w2_s0 / Z_sq)

    # Vk calculation
    den_sn = torch.clamp(1.0 - S0, min=eps)
    mu_a_norm2 = torch.sum(mu_a**2, dim=1, keepdim=True)
    num_Vk = S1a - 2.0 * torch.sum(mu_a * S2a, dim=1, keepdim=True) + mu_a_norm2 * S0
    Vk = num_Vk / den_sn

    # Vt Calculation
    # Need S1b, S2b
    # S2b = -inv_v * (y * S0 - et * sum(w^2 x))
    term_w2_x = acc_w2_x / Z_sq
    S2b = -inv_v * (y * S0 - et * term_w2_x)

    # S1b logic from original:
    # y_norm2 * S0 - 2*et*sum(w^2 y.x) + et^2 sum(w^2 ||x||^2)
    # y is constant per query, pull out
    y_norm2 = torch.sum(y**2, dim=1, keepdim=True)
    y_dot_w2x = torch.sum(y * term_w2_x, dim=1, keepdim=True)
    term_w2_x_norm = acc_w2_x_norm.unsqueeze(1) / Z_sq

    S1b = (inv_v**2) * (y_norm2 * S0 - 2.0 * et * y_dot_w2x + (et**2) * term_w2_x_norm)

    mu_b_norm2 = torch.sum(mu_b**2, dim=1, keepdim=True)
    num_Vt = S1b - 2.0 * torch.sum(mu_b * S2b, dim=1, keepdim=True) + mu_b_norm2 * S0
    Vt = num_Vt / den_sn

    # Covariance C Calculation
    # a dot z term -> scale * sum(w^2 s0.x)
    term_w2_dot = acc_w2_dot.unsqueeze(1) / Z_sq
    term_c2 = scale_factor * term_w2_dot

    # term_c1 = sum(S2a * y) -> dot product per row
    term_c1 = torch.sum(S2a * y, dim=1, keepdim=True)

    Sab = -inv_v * (term_c1 - et * term_c2)

    num_C = Sab - torch.sum(mu_a * S2b, dim=1, keepdim=True) - torch.sum(mu_b * S2a, dim=1, keepdim=True) + torch.sum(mu_a * mu_b, dim=1, keepdim=True) * S0
    C = num_C / den_sn

    # Lambda and Combination
    denom = torch.clamp(Vk + Vt - 2.0 * C, min=eps)
    lam = (Vk - C) / denom
    lam = torch.clamp(lam, 0.0, 0.95)

    return lam * score_twd + (1.0 - lam) * mu_a # mu_a is s_kss


_CE_HLSI_CACHE = {}

def _get_ce_hlsi_gate_eigenbasis(P_bar, et2, var_t):
    """
    Given the SNIS-averaged posterior Hessian P_bar [M, d, d],
    returns the gate eigenvalues and eigenvectors.

    Gate: A_bar = e^{-2t} (e^{-2t} I + v_t P_bar)^{-1}
    In eigenbasis of P_bar with eigenvalues lam_k:
        A_k = e^{-2t} / (e^{-2t} + v_t * lam_k)
    """
    # Eigendecompose and clamp to PSD
    lam, V = torch.linalg.eigh(P_bar)          # [M, d], [M, d, d]
    lam = lam.clamp(min=1e-6)                   # PSD floor

    # Gate eigenvalues
    gate_eig = et2 / (et2 + var_t * lam)        # [M, d], in [0, 1]

    return gate_eig, V


def eval_ce_hlsi_posterior_score(y, t, X_ref, log_lik_ref,
                                  P_ref, s0_post_ref):
    """
    CE-HLSI: Certainty-Equivalent Hessian-Laplace Score Identity.

    Uses a (y,t)-measurable FULL MATRIX gate built from the
    SNIS-averaged posterior Hessian P_bar = sum_i w_i P_ref_i.

    Gate:   A_bar(y,t) = e^{-2t} (e^{-2t} I + v_t P_bar)^{-1}
    Score:  s = (I - A_bar) s_TWD + A_bar s_TSI
           = s_TWD + A_bar (s_TSI - s_TWD)

    Unlike coupled HLSI (which uses per-component gates A_i that
    depend on the latent x_0^i), the CE gate is measurable w.r.t.
    (y,t) alone. This makes CE-HLSI formally unbiased for the true
    posterior score s(y,t), at the cost of losing the per-component
    coupling that gives HLSI its analytic cancellation of 1/epsilon
    singularities.

    In the near-unimodal regime (typical for stiff inverse problems),
    the SNIS weights concentrate on a single reference and
    P_bar ≈ P_dominant, so CE-HLSI collapses back to coupled HLSI.

    Parameters
    ----------
    y           : [M, d]       query points
    t           : float        diffusion time
    X_ref       : [N_ref, d]   reference points (prior samples)
    log_lik_ref : [N_ref]      log-likelihood at references
    P_ref       : [N_ref, d, d] band-gated posterior precision per reference
    s0_post_ref : [N_ref, d]   posterior score s0^post(x_ref) at references

    Returns
    -------
    score : [M, d]
    """
    if t < 1e-4:
        t = 1e-4

    et  = math.exp(-t)
    et2 = et * et
    vt  = 1.0 - math.exp(-2.0 * t)

    # ---- SNIS posterior weights ----
    w = get_posterior_snis_weights(y, t, X_ref, log_lik_ref)   # [M, N_ref]

    # ---- Tweedie score: -(y - e^{-t} E[x_0|y]) / v_t ----
    mu_x  = torch.einsum('mn,nd->md', w, X_ref)                # [M, d]
    s_twd = -(1.0 / vt) * (y - et * mu_x)

    # ---- TSI score: (1/alpha_t) E[s0^post(x_0) | y] ----
    s_tsi = (1.0 / et) * torch.einsum('mn,nd->md', w, s0_post_ref)

    # ---- SNIS-weighted mean posterior Hessian ----
    # P_bar = sum_i w_i P_ref_i,   shape [M, d, d]
    P_bar = torch.einsum('mn,nij->mij', w, P_ref)

    # ---- Gate in eigenbasis of P_bar ----
    gate_eig, V = _get_ce_hlsi_gate_eigenbasis(P_bar, et2, vt)
    # gate_eig: [M, d]   A_k = et2/(et2 + vt*lam_k)
    # V:        [M, d, d] columns = eigenvectors of P_bar

    # ---- Apply gate: s = s_twd + A_bar @ (s_tsi - s_twd) ----
    # A_bar = V diag(gate_eig) V^T
    # Efficient: V (gate_eig * (V^T @ diff))
    diff     = s_tsi - s_twd                                   # [M, d]
    diff_eig = torch.einsum('mji,mj->mi', V, diff)             # V^T @ diff  [M, d]
    A_diff   = torch.einsum('mij,mj->mi', V, gate_eig * diff_eig)  # V @ (g * V^T diff)

    return s_twd + A_diff



def get_score_wrapper(y, t, mode, X_ref, s0_post_ref, log_lik_ref,
                      P_ref=None, mu_ref=None, gated_info=None):
    """
    Score dispatch.
    Modes: 'tweedie', 'blend_posterior', 'hlsi_posterior', 'ce_hlsi'.

    The weighting choice is handled outside this function by selecting which
    log-weight bank is passed in as ``log_lik_ref`` (default likelihood L,
    WC mass proxy, or PoU-corrected mass proxy).
    """
    t_val = t.item() if isinstance(t, torch.Tensor) else float(t)
    if t_val < 1e-4:
        t_val = 1e-4

    if mode == 'tweedie':
        et = math.exp(-t_val)
        inv_v = 1.0 / (1.0 - math.exp(-2.0 * t_val))
        w = get_posterior_snis_weights(y, t_val, X_ref, log_lik_ref)
        mu_x = torch.einsum('mn,nd->md', w, X_ref)
        return -inv_v * (y - et * mu_x)

    elif mode == 'blend_posterior':
        return eval_blend_posterior_score(y, t_val, X_ref, s0_post_ref, log_lik_ref)

    elif mode == 'hlsi_posterior':
        if P_ref is None or mu_ref is None:
            raise ValueError("mode='hlsi_posterior' requires P_ref and mu_ref.")
        return eval_hlsi_posterior_score(y, t_val, X_ref, log_lik_ref,
                                         P_ref, mu_ref, gated_info=gated_info)

    elif mode == 'ce_hlsi':
        if P_ref is None:
            raise ValueError("mode='ce_hlsi' requires P_ref.")
        return eval_ce_hlsi_posterior_score(y, t_val, X_ref, log_lik_ref,
                                            P_ref, s0_post_ref)

    else:
        raise ValueError(f"Unknown mode '{mode}'.")


def compute_mean_ess(y, t, X_ref, log_lik_ref, eps=1e-30):
    """
    Mean SNIS effective sample size across query particles at diffusion time t.
    ESS(y_m, t) = 1 / sum_i w_i(y_m, t)^2, using normalized posterior weights.
    """
    w = get_posterior_snis_weights(y, t, X_ref, log_lik_ref)
    ess_per_particle = 1.0 / torch.clamp(torch.sum(w**2, dim=1), min=eps)
    return ess_per_particle.mean().item()


def run_sampler_heun(n_samples, mode, X_ref, s0_post_ref, log_lik_ref,
                     P_ref=None, mu_ref=None, gated_info=None,
                     steps=40, dim=15, log_mean_ess=False,
                     x_init=None, t_max=10.0, t_min=10**(-2.5)):
    """
    Heun sampler for the reverse-time SDE with optional warm-start state x_init.
    """
    if x_init is None:
        y = torch.randn(n_samples, dim, device=device, dtype=torch.float64)
    else:
        y = x_init.detach().clone().to(device=device, dtype=torch.float64)
        if y.ndim != 2:
            raise ValueError('x_init must have shape [n_samples, dim].')
        n_samples = y.shape[0]
        dim = y.shape[1]

    if steps <= 0:
        if log_mean_ess:
            return y, {'t': np.array([]), 'mean_ess': np.array([])}
        return y

    ts = torch.logspace(math.log10(t_max), math.log10(t_min), steps + 1,
                        device=device, dtype=torch.float64)

    ess_trace = None
    if log_mean_ess:
        ess_trace = {
            't': [ts[0].item()],
            'mean_ess': [compute_mean_ess(y, ts[0].item(), X_ref, log_lik_ref)],
        }

    for i in range(steps):
        t_cur = ts[i]
        t_next = ts[i + 1]
        dt = t_cur - t_next

        s_cur = get_score_wrapper(y, t_cur, mode, X_ref, s0_post_ref, log_lik_ref,
                                  P_ref=P_ref, mu_ref=mu_ref, gated_info=gated_info)
        d_cur = y + 2.0 * s_cur

        z = torch.randn_like(y)
        y_hat = y + d_cur * dt + math.sqrt(2.0 * dt.item()) * z

        s_next = get_score_wrapper(y_hat, t_next, mode, X_ref, s0_post_ref, log_lik_ref,
                                   P_ref=P_ref, mu_ref=mu_ref, gated_info=gated_info)
        d_next = y_hat + 2.0 * s_next

        y = y + 0.5 * (d_cur + d_next) * dt + math.sqrt(2.0 * dt.item()) * z

        if log_mean_ess:
            ess_trace['t'].append(t_next.item())
            ess_trace['mean_ess'].append(compute_mean_ess(y, t_next.item(), X_ref, log_lik_ref))

    if log_mean_ess:
        ess_trace = {k: np.array(v) for k, v in ess_trace.items()}
        return y, ess_trace

    return y


def run_mala_sampler(n_samples, prior_model, lik_model, steps=1000, dt=5e-4, burn_in=200):
    # Tuned dt=5e-4 for Navier-Stokes to ensure stability with stiffer gradients
    x = prior_model.sample(n_samples)

    log_prior = prior_model.log_prob(x)
    log_lik = lik_model.log_likelihood(x)
    score_prior = prior_model.score0(x)
    grad_lik = lik_model.grad_log_likelihood(x)

    log_post = log_prior + log_lik
    grad_log_post = score_prior + grad_lik

    accept_count = 0
    for i in range(steps):
        noise = torch.randn_like(x)
        x_prop = x + dt * grad_log_post + math.sqrt(2 * dt) * noise

        log_prior_prop = prior_model.log_prob(x_prop)
        log_lik_prop = lik_model.log_likelihood(x_prop)
        score_prior_prop = prior_model.score0(x_prop)
        grad_lik_prop = lik_model.grad_log_likelihood(x_prop)

        log_post_prop = log_prior_prop + log_lik_prop
        grad_log_post_prop = score_prior_prop + grad_lik_prop

        log_q_fwd = -torch.sum((x_prop - x - dt * grad_log_post)**2, dim=1) / (4 * dt)
        log_q_bwd = -torch.sum((x - x_prop - dt * grad_log_post_prop)**2, dim=1) / (4 * dt)

        log_alpha = log_post_prop - log_post + log_q_bwd - log_q_fwd
        accept = torch.log(torch.rand(n_samples, device=device)) < log_alpha

        x[accept] = x_prop[accept]
        log_post[accept] = log_post_prop[accept]
        grad_log_post[accept] = grad_log_post_prop[accept]

        if i >= burn_in:
            accept_count += accept.float().mean().item()

        if i % 100 == 0:
            print(f"MALA Iteration {i}/{steps}")
    print(f"MALA Acceptance: {accept_count / max(1, steps - burn_in):.2f}")
    return x

# ==========================================
# 4. EVALUATION UTILS
# ==========================================
def robust_clean_samples(samps):
    samps_np = samps.cpu().numpy() if isinstance(samps, torch.Tensor) else samps
    valid_mask = np.isfinite(samps_np).all(axis=1)
    if valid_mask.sum() < 10: return torch.tensor(samps_np[valid_mask], device=device)
    q25 = np.percentile(samps_np[valid_mask], 25, axis=0)
    q75 = np.percentile(samps_np[valid_mask], 75, axis=0)
    iqr = q75 - q25
    lower = q25 - 5.0 * iqr
    upper = q75 + 5.0 * iqr
    in_bounds = (samps_np >= lower) & (samps_np <= upper)
    valid_mask = valid_mask & in_bounds.all(axis=1)
    return torch.tensor(samps_np[valid_mask], device=device)

def sliced_wasserstein_distance(X_a, X_b, num_projections=500, p=2):
    n_a = X_a.shape[0]
    n_b = X_b.shape[0]
    if n_a > n_b:
        idx = torch.randperm(n_a)[:n_b]
        X_a = X_a[idx]
    elif n_b > n_a:
        idx = torch.randperm(n_b)[:n_a]
        X_b = X_b[idx]
    dim = X_a.shape[1]
    projections = torch.randn((num_projections, dim), device=X_a.device)
    projections = projections / torch.norm(projections, dim=1, keepdim=True)
    proj_a = torch.matmul(X_a, projections.t())
    proj_b = torch.matmul(X_b, projections.t())
    proj_a_sorted, _ = torch.sort(proj_a, dim=0)
    proj_b_sorted, _ = torch.sort(proj_b, dim=0)
    wd = torch.pow(torch.abs(proj_a_sorted - proj_b_sorted), p).mean()
    return torch.pow(wd, 1.0/p).item()

def compute_moment_errors(samples_approx, samples_ref):
    mean_approx = torch.mean(samples_approx, dim=0)
    mean_ref = torch.mean(samples_ref, dim=0)
    mean_err = torch.norm(mean_approx - mean_ref).item()

    centered_approx = samples_approx - mean_approx
    centered_ref = samples_ref - mean_ref
    cov_approx = torch.matmul(centered_approx.t(), centered_approx) / (samples_approx.shape[0] - 1)
    cov_ref = torch.matmul(centered_ref.t(), centered_ref) / (samples_ref.shape[0] - 1)
    cov_err = torch.norm(cov_approx - cov_ref).item()
    return mean_err, cov_err

def compute_mmd_rbf(X, Y, sigma=None):
    # Subsample if datasets are large to avoid O(N^2) memory issues
    n_max = 2000
    if X.shape[0] > n_max: X = X[:n_max]
    if Y.shape[0] > n_max: Y = Y[:n_max]

    # Compute pairwise distances
    dist_xx = torch.cdist(X, X, p=2)**2
    dist_yy = torch.cdist(Y, Y, p=2)**2
    dist_xy = torch.cdist(X, Y, p=2)**2

    # Median Heuristic
    if sigma is None:
        combined = torch.cat([dist_xx.view(-1), dist_yy.view(-1), dist_xy.view(-1)])
        sigma = torch.median(combined[combined > 0])
        sigma = torch.sqrt(sigma) if sigma > 0 else 1.0

    gamma = 1.0 / (2 * sigma**2)
    K_xx = torch.exp(-gamma * dist_xx)
    K_yy = torch.exp(-gamma * dist_yy)
    K_xy = torch.exp(-gamma * dist_xy)

    mmd_sq = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()
    return torch.sqrt(torch.clamp(mmd_sq, min=0.0)).item()

def rmse_vec(x_hat, x_true, eps=1e-12):
    # RMSE estimator for x_true
    return torch.sqrt(torch.mean((x_hat - x_true)**2)).item()

def rel_l2_vec(x_hat, x_true, eps=1e-12):
    num = torch.norm(x_hat - x_true).item()
    den = torch.norm(x_true).item() + eps
    return num / den


# --- ADDED KL AND KSD HELPERS ---


def compute_knn_entropy(samples, k=5):
    n, d = samples.shape
    if n <= k: return 0.0
    dists = torch.cdist(samples, samples)
    k_dists, _ = torch.topk(dists, k + 1, largest=False, dim=1)
    r_k = k_dists[:, k]
    log_vd = (d / 2.0) * math.log(math.pi) - torch.lgamma(torch.tensor(d / 2.0 + 1.0, device=samples.device))
    avg_log_dist = torch.log(r_k + 1e-10).mean()
    digamma_k = torch.digamma(torch.tensor(float(k), device=samples.device))
    entropy = d * avg_log_dist + math.log(n) - digamma_k + log_vd
    return entropy.item()

def compute_kl_divergence(samples, prior_model, lik_model):
    # KL(q || p) approx -H(q) - E_q[log p]
    # log p(x) = log p0(x) + log L(y|x) + const
    clean_x = robust_clean_samples(samples)
    if len(clean_x) < 20: return float('inf')

    # Entropy
    entropy = compute_knn_entropy(clean_x, k=5)

    # Cross Entropy
    with torch.no_grad():
        log_prior = prior_model.log_prob(clean_x)
        log_lik = lik_model.log_likelihood(clean_x)
        unnorm_log_post = log_prior + log_lik
        expected_log_p = torch.mean(unnorm_log_post).item()

    return -entropy - expected_log_p

def posterior_score_fn(x):
    # Gradient of log posterior: s0(x) + grad_lik(x)
    with torch.no_grad():
        s_prior = -x # Gaussian prior score
        s_lik = lik_model_ns.grad_log_likelihood(x) # Uses the global lik_model_ns
        return s_prior + s_lik

def compute_multiscale_ksd(samples, score_func, sigmas=(0.1, 0.2, 0.4, 0.8)):
    # Subsampling for O(N^2) cost
    N = samples.shape[0]
    if N > 1000:
        idx = torch.randperm(N)[:1000]
        samples = samples[idx]
        N = 1000

    X = samples
    D = X.shape[1]
    s = score_func(X)

    # diff: [N, N, D]
    diff = X.unsqueeze(1) - X.unsqueeze(0)
    # r2: [N, N]
    r2 = torch.sum(diff**2, dim=-1)

    ksd2 = 0.0
    for sigma in sigmas:
        K = torch.exp(-r2 / (2 * sigma**2))

        # Term 1: <s(x), s(y)> * k(x,y)
        sdot = torch.matmul(s, s.t())
        term1 = sdot * K

        # Term 2: <s(x), grad_y k> + <s(y), grad_x k>
        # grad_x k = - (x-y)/sigma^2 * K
        r_dot_sx = torch.einsum('ijd,id->ij', diff, s)
        r_dot_sy = torch.einsum('ijd,jd->ij', diff, s)

        # Note: diff_ij = x_i - x_j.
        # grad_xi K = -diff_ij / sigma^2 * K
        # grad_xj K = +diff_ij / sigma^2 * K
        term2 = (r_dot_sx - r_dot_sy) / (sigma**2) * K # double check sign logic standard Stein derivation

        # Term 3: div_x div_y k
        term3 = (D / (sigma**2) - r2 / (sigma**4)) * K

        U = term1 + term2 + term3
        ksd2 += torch.sum(U) / (N * N)

    return ksd2.item() / len(sigmas)


# ==========================================
# 5. PCA VISUALIZATION (updated w/ HLSI)
# ==========================================
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_pca_histograms(samples_dict, alpha_true):
    """
    PCA basis, plot limits, and histogram brightness normalization are all taken
    from the MALA reference when available. This prevents a blown-up sampler from
    setting the plotting scale for the other methods.
    """

    # --- Choose latent dim consistently ---
    if "ACTIVE_DIM" in globals():
        d_lat = int(ACTIVE_DIM)
    else:
        # fall back: infer from first available samples
        any_key = next(iter(samples_dict.keys()))
        d_lat = int(robust_clean_samples(samples_dict[any_key]).shape[1])

    alpha_true = np.asarray(alpha_true).reshape(-1)[:d_lat]

    # --- Pick PCA anchor method robustly ---
    anchor_candidates = ["mala", "blend_posterior", "hlsi_posterior", "wc_hlsi", "tweedie"]
    anchor = None
    for a in anchor_candidates:
        if a in samples_dict:
            anchor = a
            break
    if anchor is None:
        raise ValueError("No samples found in samples_dict to compute PCA basis.")

    anchor_data = robust_clean_samples(samples_dict[anchor])
    if anchor_data.shape[0] < 10:
        raise ValueError(f"Not enough valid samples in anchor method '{anchor}' for PCA.")

    # Center using anchor mean
    mean_anchor = torch.mean(anchor_data[:, :d_lat], dim=0)
    centered_anchor = anchor_data[:, :d_lat] - mean_anchor

    # PCA via SVD (use linalg.svd; torch.svd is deprecated-ish)
    # centered_anchor: (N, d) = U * diag(S) * Vh, where Vh: (d,d)
    U, S, Vh = torch.linalg.svd(centered_anchor, full_matrices=False)
    V = Vh.T  # columns are principal directions, shape (d_lat, d_lat)

    # Identify disjoint pairs of PCs
    pairs = [(0, 1), (2, 3)]

    # Include HLSI + WC-HLSI in the plot set
    methods = ['mala', 'tweedie', 'blend_posterior', 'hlsi_posterior', 'wc_hlsi']
    titles = {
        'mala': 'MALA',
        'tweedie': 'Tweedie',
        'blend_posterior': 'Blend',
        'hlsi_posterior': 'HLSI',
        'wc_hlsi': 'WC-HLSI',
    }

    # Filter methods to those that exist
    methods = [m for m in methods if m in samples_dict]
    if len(methods) == 0:
        raise ValueError("None of the requested methods exist in samples_dict.")

    fig, axes = plt.subplots(len(pairs), len(methods), figsize=(5 * len(methods), 5 * len(pairs)))
    if len(pairs) == 1:
        axes = np.expand_dims(axes, axis=0)
    if len(methods) == 1:
        axes = np.expand_dims(axes, axis=1)

    for row_idx, (d1, d2) in enumerate(pairs):
        if d2 >= V.shape[1]:
            d1, d2 = 0, 1

        v1 = V[:, d1]
        v2 = V[:, d2]

        # Project true alpha (centered with anchor mean!)
        true_cent = torch.tensor(alpha_true, device=mean_anchor.device, dtype=mean_anchor.dtype) - mean_anchor
        t1 = torch.dot(true_cent, v1).item()
        t2 = torch.dot(true_cent, v2).item()

        # Determine plot limits from the MALA/anchor projections (robust)
        proj_anchor_1 = torch.matmul(centered_anchor, v1).detach().cpu().numpy()
        proj_anchor_2 = torch.matmul(centered_anchor, v2).detach().cpu().numpy()
        q01_x, q99_x = np.percentile(proj_anchor_1, [1, 99])
        q01_y, q99_y = np.percentile(proj_anchor_2, [1, 99])
        span_x = max(q99_x - q01_x, 1e-12)
        span_y = max(q99_y - q01_y, 1e-12)
        pad = 0.5
        xlims = [q01_x - pad * span_x, q99_x + pad * span_x]
        ylims = [q01_y - pad * span_y, q99_y + pad * span_y]

        # Histogram brightness normalization is also referenced to MALA/anchor.
        ref_hist, _, _ = np.histogram2d(
            proj_anchor_1,
            proj_anchor_2,
            bins=60,
            range=[xlims, ylims],
            density=True,
        )
        hist_vmax = max(float(np.nanmax(ref_hist)), 1e-12)

        for col_idx, m in enumerate(methods):
            ax = axes[row_idx, col_idx]
            ax.set_xticks([])
            ax.set_yticks([])

            samps = robust_clean_samples(samples_dict[m])
            if samps.shape[0] < 10:
                ax.set_title(f"{titles[m]} (unstable)", fontsize=16)
                ax.axis("off")
                continue

            centered = samps[:, :d_lat] - mean_anchor
            p1 = torch.matmul(centered, v1).detach().cpu().numpy()
            p2 = torch.matmul(centered, v2).detach().cpu().numpy()

            ax.hist2d(
                p1, p2,
                bins=60,
                range=[xlims, ylims],
                cmap='inferno',
                density=True,
                vmax=hist_vmax
            )
            ax.scatter(t1, t2, c='cyan', marker='x', s=200, linewidth=4, label='True $\\alpha$')

            if row_idx == 0:
                ax.set_title(f"{titles[m]}", fontsize=18)
            if col_idx == 0:
                ax.set_ylabel(f"PC {d1+1} vs PC {d2+1}", fontsize=18)
            if row_idx == 0 and col_idx == 0:
                ax.legend(fontsize=14)

    plt.suptitle(f"PCA of posterior samples (anchor={anchor}, dim={d_lat})", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()


# ==========================================
# 6. EXECUTION (MODULARIZED SAMPLER CONFIGS)
# ==========================================
from collections import OrderedDict

def _normalize_sampler_key(name):
    if name is None:
        raise ValueError('Sampler/init name cannot be None.')
    key = str(name).strip().lower().replace('_', ' ').replace('-', ' ')
    return ' '.join(key.split())


INIT_ALIASES = {
    _normalize_sampler_key('prior'): 'prior',
    _normalize_sampler_key('tweedie'): 'tweedie',
    _normalize_sampler_key('blend'): 'blend_posterior',
    _normalize_sampler_key('blend_posterior'): 'blend_posterior',
    _normalize_sampler_key('hlsi'): 'hlsi_posterior',
    _normalize_sampler_key('hlsi_posterior'): 'hlsi_posterior',
    _normalize_sampler_key('ce-hlsi'): 'ce_hlsi',
    _normalize_sampler_key('ce_hlsi'): 'ce_hlsi',
    _normalize_sampler_key('ce hlsi'): 'ce_hlsi',
}


INIT_DISPLAY_NAMES = {
    'prior': 'Prior',
    'tweedie': 'Tweedie',
    'blend_posterior': 'Blend',
    'hlsi_posterior': 'HLSI',
    'ce_hlsi': 'CE-HLSI',
    'mala': 'MALA',
}


INIT_WEIGHT_ALIASES = {
    _normalize_sampler_key('l'): 'L',
    _normalize_sampler_key('likelihood'): 'L',
    _normalize_sampler_key('default'): 'L',
    _normalize_sampler_key('posterior'): 'L',
    _normalize_sampler_key('wc'): 'WC',
    _normalize_sampler_key('weight corrected'): 'WC',
    _normalize_sampler_key('pou'): 'PoU',
    _normalize_sampler_key('po u'): 'PoU',
    _normalize_sampler_key('p ou'): 'PoU',
    _normalize_sampler_key('partition of unity'): 'PoU',
}

INIT_WEIGHT_BANK_KEYS = {
    'L': 'log_lik_ref',
    'WC': 'log_mass_ref',
    'PoU': 'log_pou_ref',
}


LEGACY_INIT_SPECS = {
    _normalize_sampler_key('wc-hlsi'): ('hlsi_posterior', 'WC'),
    _normalize_sampler_key('wc_hlsi'): ('hlsi_posterior', 'WC'),
    _normalize_sampler_key('wc hlsi'): ('hlsi_posterior', 'WC'),
    _normalize_sampler_key('pou-hlsi'): ('hlsi_posterior', 'PoU'),
    _normalize_sampler_key('pou_hlsi'): ('hlsi_posterior', 'PoU'),
    _normalize_sampler_key('pou hlsi'): ('hlsi_posterior', 'PoU'),
    _normalize_sampler_key('ce-wc-hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('ce_wc_hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('ce wc hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('wc-ce-hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('wc_ce_hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('wc ce hlsi'): ('ce_hlsi', 'WC'),
    _normalize_sampler_key('ce-pou-hlsi'): ('ce_hlsi', 'PoU'),
    _normalize_sampler_key('ce_pou_hlsi'): ('ce_hlsi', 'PoU'),
    _normalize_sampler_key('ce pou hlsi'): ('ce_hlsi', 'PoU'),
}


def canonicalize_init_name(name):
    key = _normalize_sampler_key(name)
    if key not in INIT_ALIASES:
        raise ValueError(f"Unknown sampler/init family: {name}")
    return INIT_ALIASES[key]


def parse_init_spec(name):
    key = _normalize_sampler_key(name)
    if key in LEGACY_INIT_SPECS:
        return LEGACY_INIT_SPECS[key]
    return canonicalize_init_name(name), None


def canonicalize_init_weights(name):
    if name is None:
        return 'L'
    key = _normalize_sampler_key(name)
    if key not in INIT_WEIGHT_ALIASES:
        raise ValueError(f"Unknown init_weights mode: {name}")
    return INIT_WEIGHT_ALIASES[key]


def format_sampler_display_name(init_mode, init_weights='L'):
    if init_mode == 'prior':
        return 'Prior'
    base = INIT_DISPLAY_NAMES.get(init_mode, str(init_mode))
    if init_weights == 'L':
        return base
    if init_mode == 'hlsi_posterior' and init_weights == 'WC':
        return 'WC-HLSI'
    if init_mode == 'hlsi_posterior' and init_weights == 'PoU':
        return 'PoU-HLSI'
    if init_mode == 'ce_hlsi' and init_weights == 'WC':
        return 'CE-WC-HLSI'
    if init_mode == 'ce_hlsi' and init_weights == 'PoU':
        return 'CE-PoU-HLSI'
    return f"{base} [{init_weights}]"


def pretty_sampler_name(name):
    if name in {'mala', 'MALA'}:
        return 'MALA'
    init_mode, implied_weights = parse_init_spec(name)
    if init_mode == 'prior':
        return 'Prior'
    return format_sampler_display_name(init_mode, implied_weights if implied_weights is not None else 'L')


def get_sampler_precomp_bank(precomp, init_mode, init_weights='L'):
    init_weights = canonicalize_init_weights(init_weights)
    if init_weights == 'PoU' and init_mode in {'hlsi_posterior', 'ce_hlsi'}:
        if not all(k in precomp for k in ['s0_pou_ref', 'P_pou_ref', 'mu_pou_ref', 'gated_info_pou']):
            raise KeyError('PoU local bank is missing one of s0_pou_ref, P_pou_ref, mu_pou_ref, gated_info_pou.')
        bank = dict(precomp)
        bank['s0_post_ref'] = precomp['s0_pou_ref']
        bank['P_ref'] = precomp['P_pou_ref']
        bank['mu_ref'] = precomp['mu_pou_ref']
        bank['gated_info'] = precomp['gated_info_pou']
        return bank
    return precomp


def get_sampler_log_weights(init_weights, bank):
    init_weights = canonicalize_init_weights(init_weights)
    bank_key = INIT_WEIGHT_BANK_KEYS[init_weights]
    if bank_key not in bank:
        raise KeyError(
            f"Reference bank is missing '{bank_key}'. Available keys: {sorted(bank.keys())}"
        )
    return bank[bank_key]


def get_sampler_log_weight_name(init_weights):
    init_weights = canonicalize_init_weights(init_weights)
    return INIT_WEIGHT_BANK_KEYS[init_weights]


def choose_reference_key(samples_dict, sampler_run_info=None, preferred=None):
    if preferred is not None and preferred in samples_dict:
        return preferred

    if sampler_run_info is not None:
        for label, info in sampler_run_info.items():
            if info.get('is_reference', False) and label in samples_dict:
                return label
        for label, info in sampler_run_info.items():
            if info.get('mala_steps', 0) > 0 and label in samples_dict:
                return label

    for label in samples_dict:
        return label

    raise ValueError('No sampler outputs available to choose a reference key.')


def normalize_sampler_config(label, config, default_n_samples, default_dim):
    cfg = dict(config)
    cfg.setdefault('init', 'prior')

    init_raw = cfg['init']
    if str(init_raw).lower() == 'prior':
        cfg['init'] = 'prior'
        implied_weights = None
    else:
        cfg['init'], implied_weights = parse_init_spec(init_raw)

    if cfg['init'] == 'prior':
        cfg['init_weights'] = 'prior'
    else:
        cfg['init_weights'] = canonicalize_init_weights(
            cfg.get('init_weights', implied_weights if implied_weights is not None else 'L')
        )

    cfg.setdefault('display_name', format_sampler_display_name(
        cfg['init'], 'L' if cfg['init'] == 'prior' else cfg['init_weights']))
    cfg.setdefault('init_steps', 0 if cfg['init'] == 'prior' else 200)
    cfg.setdefault('init_tmax', 10.0)
    cfg.setdefault('init_tmin', 10 ** (-2.5))
    cfg.setdefault('log_mean_ess', cfg['init'] != 'prior')
    cfg.setdefault('n_samples', default_n_samples)
    cfg.setdefault('dim', default_dim)
    cfg.setdefault('mala_steps', 0)
    cfg.setdefault('mala_burnin', 0)
    cfg.setdefault('mala_dt', 5e-4)
    cfg.setdefault('is_reference', False)

    if cfg['init'] == 'prior' and cfg['init_steps'] != 0:
        print(f"[normalize_sampler_config] '{label}': init='prior' ignores "
              f"init_steps={cfg['init_steps']}; setting to 0.")
        cfg['init_steps'] = 0

    if cfg['mala_steps'] <= 0:
        cfg['mala_steps'] = 0
        cfg['mala_burnin'] = 0

    return cfg


def run_sampler_heun(n_samples, mode, X_ref, s0_post_ref, log_lik_ref,
                     P_ref=None, mu_ref=None, gated_info=None,
                     steps=40, dim=15, log_mean_ess=False,
                     x_init=None, t_max=10.0, t_min=10**(-2.5)):
    """
    Heun sampler for the reverse-time SDE with optional warm-start state x_init.
    """
    if x_init is None:
        y = torch.randn(n_samples, dim, device=device, dtype=torch.float64)
    else:
        y = x_init.detach().clone().to(device=device, dtype=torch.float64)
        if y.ndim != 2:
            raise ValueError('x_init must have shape [n_samples, dim].')
        n_samples = y.shape[0]
        dim = y.shape[1]

    if steps <= 0:
        if log_mean_ess:
            return y, {'t': np.array([]), 'mean_ess': np.array([])}
        return y

    ts = torch.logspace(math.log10(t_max), math.log10(t_min), steps + 1,
                        device=device, dtype=torch.float64)

    ess_trace = None
    if log_mean_ess:
        ess_trace = {
            't': [ts[0].item()],
            'mean_ess': [compute_mean_ess(y, ts[0].item(), X_ref, log_lik_ref)],
        }

    for i in range(steps):
        t_cur = ts[i]
        t_next = ts[i + 1]
        dt = t_cur - t_next

        s_cur = get_score_wrapper(y, t_cur, mode, X_ref, s0_post_ref, log_lik_ref,
                                  P_ref=P_ref, mu_ref=mu_ref, gated_info=gated_info)
        d_cur = y + 2.0 * s_cur

        z = torch.randn_like(y)
        y_hat = y + d_cur * dt + math.sqrt(2.0 * dt.item()) * z

        s_next = get_score_wrapper(y_hat, t_next, mode, X_ref, s0_post_ref, log_lik_ref,
                                   P_ref=P_ref, mu_ref=mu_ref, gated_info=gated_info)
        d_next = y_hat + 2.0 * s_next

        y = y + 0.5 * (d_cur + d_next) * dt + math.sqrt(2.0 * dt.item()) * z

        if log_mean_ess:
            ess_trace['t'].append(t_next.item())
            ess_trace['mean_ess'].append(compute_mean_ess(y, t_next.item(), X_ref, log_lik_ref))

    if log_mean_ess:
        ess_trace = {k: np.array(v) for k, v in ess_trace.items()}
        return y, ess_trace

    return y


def run_mala_sampler(n_samples, prior_model, lik_model, steps=1000, dt=5e-4,
                     burn_in=200, x_init=None, verbose=True, return_info=False):
    """
    MALA with optional warm start x_init. If x_init is provided, n_samples is inferred from it.
    """
    if x_init is None:
        x = prior_model.sample(n_samples)
    else:
        x = x_init.detach().clone().to(device=device, dtype=torch.float64)
        if x.ndim != 2:
            raise ValueError('x_init must have shape [n_samples, dim].')
        n_samples = x.shape[0]

    log_prior = prior_model.log_prob(x)
    log_lik = lik_model.log_likelihood(x)
    score_prior = prior_model.score0(x)
    grad_lik = lik_model.grad_log_likelihood(x)

    log_post = log_prior + log_lik
    grad_log_post = score_prior + grad_lik

    accept_count = 0.0
    denom_accept = max(1, steps - burn_in)

    for i in range(steps):
        noise = torch.randn_like(x)
        x_prop = x + dt * grad_log_post + math.sqrt(2.0 * dt) * noise

        log_prior_prop = prior_model.log_prob(x_prop)
        log_lik_prop = lik_model.log_likelihood(x_prop)
        score_prior_prop = prior_model.score0(x_prop)
        grad_lik_prop = lik_model.grad_log_likelihood(x_prop)

        log_post_prop = log_prior_prop + log_lik_prop
        grad_log_post_prop = score_prior_prop + grad_lik_prop

        log_q_fwd = -torch.sum((x_prop - x - dt * grad_log_post) ** 2, dim=1) / (4.0 * dt)
        log_q_bwd = -torch.sum((x - x_prop - dt * grad_log_post_prop) ** 2, dim=1) / (4.0 * dt)

        log_alpha = log_post_prop - log_post + log_q_bwd - log_q_fwd
        accept = torch.log(torch.rand(n_samples, device=device)) < log_alpha

        x[accept] = x_prop[accept]
        log_post[accept] = log_post_prop[accept]
        grad_log_post[accept] = grad_log_post_prop[accept]

        if i >= burn_in:
            accept_count += accept.float().mean().item()

        if verbose and (i % 100 == 0):
            print(f"MALA Iteration {i}/{steps}")

    accept_rate = accept_count / denom_accept
    if verbose:
        print(f"MALA Acceptance: {accept_rate:.2f}")

    info = {
        'acceptance_post_burnin': accept_rate,
        'steps': steps,
        'burn_in': burn_in,
        'dt': dt,
    }
    if return_info:
        return x, info
    return x


def plot_mean_ess_logs(ess_logs_dict, display_names=None):
    if len(ess_logs_dict) == 0:
        print('\n=== Mean ESS vs t ===')
        print('No ESS traces were requested.')
        return

    plt.figure(figsize=(8, 5))
    for label, trace in ess_logs_dict.items():
        if trace is None or len(trace.get('t', [])) == 0:
            continue
        title = display_names.get(label, label) if display_names is not None else label
        t_vec = trace['t']
        ess_vec = trace['mean_ess']
        order = np.argsort(t_vec)
        plt.plot(t_vec[order], ess_vec[order], marker='o', linewidth=2, label=title)
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Diffusion time t')
    plt.ylabel('Mean ESS across particles')
    plt.title('Mean ESS vs diffusion time t')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Override the earlier fixed-method PCA plotting with a dynamic version.
def plot_pca_histograms(samples_dict, alpha_true, display_names=None, reference_key=None):
    if len(samples_dict) == 0:
        raise ValueError('samples_dict is empty.')

    if display_names is None:
        display_names = {k: k for k in samples_dict.keys()}

    if "ACTIVE_DIM" in globals():
        d_lat = int(ACTIVE_DIM)
    else:
        any_key = next(iter(samples_dict.keys()))
        d_lat = int(robust_clean_samples(samples_dict[any_key]).shape[1])

    alpha_true = np.asarray(alpha_true).reshape(-1)[:d_lat]

    anchor = reference_key if reference_key in samples_dict else next(iter(samples_dict.keys()))
    anchor_data = robust_clean_samples(samples_dict[anchor])
    if anchor_data.shape[0] < 10:
        raise ValueError(f"Not enough valid samples in anchor method '{anchor}' for PCA.")

    mean_anchor = torch.mean(anchor_data[:, :d_lat], dim=0)
    centered_anchor = anchor_data[:, :d_lat] - mean_anchor
    U, S, Vh = torch.linalg.svd(centered_anchor, full_matrices=False)
    V = Vh.T

    pairs = [(0, 1)]
    if V.shape[1] >= 4:
        pairs.append((2, 3))

    methods = list(samples_dict.keys())
    fig, axes = plt.subplots(len(pairs), len(methods), figsize=(5 * len(methods), 5 * len(pairs)))
    if len(pairs) == 1:
        axes = np.expand_dims(axes, axis=0)
    if len(methods) == 1:
        axes = np.expand_dims(axes, axis=1)

    for row_idx, (d1, d2) in enumerate(pairs):
        v1 = V[:, d1]
        v2 = V[:, d2]

        true_cent = torch.tensor(alpha_true, device=mean_anchor.device, dtype=mean_anchor.dtype) - mean_anchor
        t1 = torch.dot(true_cent, v1).item()
        t2 = torch.dot(true_cent, v2).item()

        proj_anchor_1 = torch.matmul(centered_anchor, v1).detach().cpu().numpy()
        proj_anchor_2 = torch.matmul(centered_anchor, v2).detach().cpu().numpy()
        q01_x, q99_x = np.percentile(proj_anchor_1, [1, 99])
        q01_y, q99_y = np.percentile(proj_anchor_2, [1, 99])
        span_x = max(q99_x - q01_x, 1e-12)
        span_y = max(q99_y - q01_y, 1e-12)
        pad = 0.5
        xlims = [q01_x - pad * span_x, q99_x + pad * span_x]
        ylims = [q01_y - pad * span_y, q99_y + pad * span_y]

        ref_hist, _, _ = np.histogram2d(
            proj_anchor_1,
            proj_anchor_2,
            bins=60,
            range=[xlims, ylims],
            density=True,
        )
        hist_vmax = max(float(np.nanmax(ref_hist)), 1e-12)

        for col_idx, label in enumerate(methods):
            ax = axes[row_idx, col_idx]
            ax.set_xticks([])
            ax.set_yticks([])

            samps = robust_clean_samples(samples_dict[label])
            if samps.shape[0] < 10:
                ax.set_title(f"{display_names.get(label, label)} (unstable)", fontsize=16)
                ax.axis('off')
                continue

            centered = samps[:, :d_lat] - mean_anchor
            p1 = torch.matmul(centered, v1).detach().cpu().numpy()
            p2 = torch.matmul(centered, v2).detach().cpu().numpy()

            ax.hist2d(
                p1, p2,
                bins=60,
                range=[xlims, ylims],
                cmap='inferno',
                density=True,
                vmax=hist_vmax,
            )
            ax.scatter(t1, t2, c='cyan', marker='x', s=200, linewidth=4, label='True $\\alpha$')

            if row_idx == 0:
                ax.set_title(display_names.get(label, label), fontsize=18)
            if col_idx == 0:
                ax.set_ylabel(f"PC {d1+1} vs PC {d2+1}", fontsize=18)
            if row_idx == 0 and col_idx == 0:
                ax.legend(fontsize=14)

    plt.suptitle(f"PCA of posterior samples (anchor={display_names.get(anchor, anchor)}, dim={d_lat})", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()


print(f"--- Setting up {ACTIVE_DIM}D Navier-Stokes Inverse Problem ---")

import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 1. Ground Truth
np.random.seed(42)
alpha_true_np = np.random.randn(ACTIVE_DIM)
alpha_true_np = alpha_true_np * 0.5
y_clean = solve_forward(jnp.array(alpha_true_np))
y_obs_np = y_clean + np.random.normal(0, NOISE_STD, size=y_clean.shape)

# 2. Models
prior_model_ns = GaussianPrior(dim=ACTIVE_DIM)
lik_model_ns = NSLikelihood(y_obs_np, NOISE_STD)

POU_QUERY_BLOCK = 128
POU_REF_BLOCK = 512


def _compute_log_pou_overlap_penalty(X_ref, mu_ref, eigvecs, prec_eig, trusted,
                                     label='base', query_block=POU_QUERY_BLOCK,
                                     ref_block=POU_REF_BLOCK):
    """
    Practical PoU correction for the WC masses:

        log m_i^{PoU} ≈ log mass_i^WC - log sum_j H(mu_i; j),

    where H(.; j) is the local Gaussian window centered at x_j with the same
    band-gated precision used by the HLSI geometry. We only change the global
    mixing weights here; the particle-wise score signal remains whatever the
    sampler config selects (HLSI, CE-HLSI, etc.).
    """
    n_ref = X_ref.shape[0]
    work_device = X_ref.device
    work_dtype = X_ref.dtype

    eig_for_logdet = torch.where(
        trusted, torch.clamp(prec_eig, min=1e-30), torch.ones_like(prec_eig)
    )
    log_window_norm = 0.5 * torch.sum(torch.log(eig_for_logdet), dim=1)

    print(f"  [{label}] Computing PoU overlap penalties with query_block={query_block}, ref_block={ref_block}...")
    t0_pou = time.time()

    log_overlap = torch.empty((n_ref,), device=work_device, dtype=work_dtype)
    n_query_blocks = (n_ref + query_block - 1) // query_block

    for qb, q0 in enumerate(range(0, n_ref, query_block)):
        q1 = min(q0 + query_block, n_ref)
        mu_block = mu_ref[q0:q1]
        block_max = torch.full((q1 - q0,), -float('inf'), device=work_device, dtype=work_dtype)
        block_sumexp = torch.zeros((q1 - q0,), device=work_device, dtype=work_dtype)

        for r0 in range(0, n_ref, ref_block):
            r1 = min(r0 + ref_block, n_ref)
            x_block = X_ref[r0:r1]
            V_block = eigvecs[r0:r1]
            lam_block = prec_eig[r0:r1]
            trust_block = trusted[r0:r1]
            log_norm_block = log_window_norm[r0:r1]

            diff = mu_block.unsqueeze(1) - x_block.unsqueeze(0)              # [Q, R, d]
            diff_eig = torch.einsum('qrd,rdk->qrk', diff, V_block)            # [Q, R, d]
            quad = torch.sum(diff_eig.pow(2) * lam_block.unsqueeze(0), dim=2) # [Q, R]
            log_window = log_norm_block.unsqueeze(0) - 0.5 * quad

            # Directions outside the trusted band contribute zero precision, which is
            # already reflected by lam_block = 0 there via prec_eig.
            current_max, _ = torch.max(log_window, dim=1)
            new_max = torch.maximum(block_max, current_max)

            block_sumexp = (
                block_sumexp * torch.exp(block_max - new_max)
                + torch.sum(torch.exp(log_window - new_max.unsqueeze(1)), dim=1)
            )
            block_max = new_max

        log_overlap[q0:q1] = block_max + torch.log(torch.clamp(block_sumexp, min=1e-300))
        if (qb + 1) % max(1, math.ceil(n_query_blocks / 10)) == 0 or (qb + 1) == n_query_blocks:
            print(f"    [{label}] PoU overlap block {qb + 1}/{n_query_blocks} complete")

    # mild floor for numerical sanity: overlap >= self-window contribution
    log_self = torch.clamp(log_window_norm, min=-1e30)
    log_overlap = torch.maximum(log_overlap, log_self)

    print(f"  [{label}] PoU overlap time: {time.time() - t0_pou:.2f}s")
    return log_overlap


def _compute_log_pou_denom_grad_hess_at_points(points, X_ref, eigvecs, prec_eig, trusted,
                                              label='base', query_block=POU_QUERY_BLOCK,
                                              ref_block=POU_REF_BLOCK):
    """
    Evaluate the undiffused PoU denominator

        Hbar(x) = sum_j H_j(x)

    together with its gradient and Hessian at a batch of query points. This
    builds the Stage-A PoU-adjusted local bank
        s_{i,0}^{PoU}, P_i^{PoU}, mu_i^{PoU}, m_i^{PoU}.
    """
    n_query, d = points.shape
    work_device = points.device
    work_dtype = points.dtype

    eig_for_logdet = torch.where(
        trusted, torch.clamp(prec_eig, min=1e-30), torch.ones_like(prec_eig)
    )
    log_window_norm = 0.5 * torch.sum(torch.log(eig_for_logdet), dim=1)

    print(f"  [{label}] Computing PoU denominator grad/Hess at query_block={query_block}, ref_block={ref_block}...")
    t0_pou = time.time()

    log_denom = torch.empty((n_query,), device=work_device, dtype=work_dtype)
    grad_log_denom = torch.empty((n_query, d), device=work_device, dtype=work_dtype)
    hess_log_denom = torch.empty((n_query, d, d), device=work_device, dtype=work_dtype)
    n_query_blocks = (n_query + query_block - 1) // query_block

    for qb, q0 in enumerate(range(0, n_query, query_block)):
        q1 = min(q0 + query_block, n_query)
        pts = points[q0:q1]
        Q = pts.shape[0]

        block_max = torch.full((Q,), -float('inf'), device=work_device, dtype=work_dtype)
        block_sumexp = torch.zeros((Q,), device=work_device, dtype=work_dtype)
        grad_num = torch.zeros((Q, d), device=work_device, dtype=work_dtype)
        second_num = torch.zeros((Q, d, d), device=work_device, dtype=work_dtype)

        for r0 in range(0, X_ref.shape[0], ref_block):
            r1 = min(r0 + ref_block, X_ref.shape[0])
            X_block = X_ref[r0:r1]
            V_block = eigvecs[r0:r1]
            lam_block = prec_eig[r0:r1]
            log_norm_block = log_window_norm[r0:r1]

            diff = pts.unsqueeze(1) - X_block.unsqueeze(0)
            diff_eig = torch.einsum('qrd,rdk->qrk', diff, V_block)
            quad = torch.sum(diff_eig.pow(2) * lam_block.unsqueeze(0), dim=2)
            log_H = log_norm_block.unsqueeze(0) - 0.5 * quad
            H_grad = -torch.einsum('qrd,rdk,rjk->qrj', diff, V_block * lam_block.unsqueeze(-1), V_block)
            H_hess = -torch.einsum('rdi,rd,rdj->rij', V_block, lam_block, V_block)

            local_max = torch.max(log_H, dim=1).values
            new_max = torch.maximum(block_max, local_max)

            block_sumexp = (
                block_sumexp * torch.exp(block_max - new_max)
                + torch.sum(torch.exp(log_H - new_max.unsqueeze(1)), dim=1)
            )
            grad_num = (
                grad_num * torch.exp((block_max - new_max).unsqueeze(1))
                + torch.einsum('qr,qrd->qd', torch.exp(log_H - new_max.unsqueeze(1)), H_grad)
            )
            second_num = (
                second_num * torch.exp((block_max - new_max).view(Q, 1, 1))
                + torch.einsum('qr,qrij->qij', torch.exp(log_H - new_max.unsqueeze(1)),
                               H_hess.unsqueeze(0) + torch.einsum('qri,qrj->qrij', H_grad, H_grad))
            )
            block_max = new_max

            del X_block, V_block, lam_block, log_norm_block, diff, diff_eig, quad, log_H, H_grad, H_hess, local_max, new_max

        Z = torch.clamp(block_sumexp, min=1e-300)
        grad_log = grad_num / Z.unsqueeze(1)
        second_over_Z = second_num / Z.view(Q, 1, 1)
        hess_log = second_over_Z - torch.einsum('qi,qj->qij', grad_log, grad_log)
        hess_log = 0.5 * (hess_log + hess_log.transpose(-1, -2))

        log_denom[q0:q1] = block_max + torch.log(Z)
        grad_log_denom[q0:q1] = grad_log
        hess_log_denom[q0:q1] = hess_log

        if (qb + 1) % max(1, n_query_blocks // 10) == 0 or q1 == n_query:
            print(f"    [{label}] PoU grad/Hess block {qb + 1}/{n_query_blocks} complete")

        del pts, block_max, block_sumexp, grad_num, second_num, Z, grad_log, second_over_Z, hess_log
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"  [{label}] PoU denominator grad/Hess time: {time.time() - t0_pou:.2f}s")
    return log_denom, grad_log_denom, hess_log_denom


# 3. Precompute reference quantities once
N_REF_NS = 10000

print(f"Generating {N_REF_NS} Prior Samples...")
t0 = time.time()
X_ref_ns = prior_model_ns.sample(N_REF_NS)

with torch.no_grad():
    s0_prior_ns = prior_model_ns.score0(X_ref_ns)
    log_prior_ns = prior_model_ns.log_prob(X_ref_ns)

    print('Calculating Likelihoods / Grads / Hessians (Batched)...')
    batch_size = 500
    log_lik_list = []
    grad_lik_list = []
    hess_lik_list = []

    for i in range(0, N_REF_NS, batch_size):
        batch = X_ref_ns[i:i+batch_size]
        log_lik_list.append(lik_model_ns.log_likelihood(batch))
        grad_lik_list.append(lik_model_ns.grad_log_likelihood(batch))
        hess_lik_list.append(lik_model_ns.hess_log_likelihood(batch))

    log_lik_ns = torch.cat(log_lik_list, dim=0)
    grad_lik_ns = torch.cat(grad_lik_list, dim=0)
    hess_lik_ns = torch.cat(hess_lik_list, dim=0)

    s0_post_ns = s0_prior_ns + grad_lik_ns

    d = X_ref_ns.shape[1]
    I = torch.eye(d, device=device, dtype=torch.float64).unsqueeze(0)

    P_raw_ns = I - hess_lik_ns
    P_raw_ns = 0.5 * (P_raw_ns + P_raw_ns.transpose(-1, -2))

    print(f"  Spectral band gating: [{HESS_MIN:.1e}, {HESS_MAX:.1e}]")
    eigvals_ns, eigvecs_ns = torch.linalg.eigh(P_raw_ns)
    trusted_ns = (eigvals_ns >= HESS_MIN) & (eigvals_ns <= HESS_MAX)

    n_below = (eigvals_ns < HESS_MIN).float().sum(dim=1).mean().item()
    n_above = (eigvals_ns > HESS_MAX).float().sum(dim=1).mean().item()
    n_band = trusted_ns.float().sum(dim=1).mean().item()
    print(f"    mean {n_band:.1f} in-band, {n_below:.1f} too-sloppy, {n_above:.1f} too-stiff (of {d})")

    prec_eig_ns = torch.where(trusted_ns, eigvals_ns, torch.zeros_like(eigvals_ns))
    P_ref_ns = torch.einsum('nij,nj,nkj->nik', eigvecs_ns, prec_eig_ns, eigvecs_ns)
    P_ref_ns = P_ref_ns + 1e-6 * I

    s_eig = (eigvecs_ns.mT @ s0_post_ns.unsqueeze(-1)).squeeze(-1)
    delta_eig = torch.where(
        trusted_ns,
        s_eig / torch.clamp(prec_eig_ns, min=HESS_MIN),
        torch.zeros_like(s_eig),
    )
    delta = (eigvecs_ns @ delta_eig.unsqueeze(-1)).squeeze(-1)
    mu_ref_ns = X_ref_ns + delta

    gated_info_ns = {
        'eigvecs': eigvecs_ns,
        'eigvals': prec_eig_ns,
        'trusted': trusted_ns,
    }

    log_post_x = log_prior_ns + log_lik_ns
    quad_gain = 0.5 * torch.sum(s0_post_ns * delta, dim=1)
    eig_for_logdet = torch.where(trusted_ns, eigvals_ns, torch.ones_like(eigvals_ns))
    logdet_P = torch.sum(torch.log(torch.clamp(eig_for_logdet, min=1e-30)), dim=1)
    log_mass_ns = (log_post_x + quad_gain) - 0.5 * logdet_P

    log_window_norm_ns = 0.5 * torch.sum(torch.log(torch.clamp(eig_for_logdet, min=1e-30)), dim=1)
    P_window_ref_ns = torch.einsum('nij,nj,nkj->nik', eigvecs_ns, prec_eig_ns, eigvecs_ns)

    log_pou_denom_ref_ns, grad_log_pou_denom_ref_ns, hess_log_pou_denom_ref_ns = _compute_log_pou_denom_grad_hess_at_points(
        X_ref_ns, X_ref_ns, eigvecs_ns, prec_eig_ns, trusted_ns, label='NS'
    )

    s0_pou_ns = s0_post_ns - grad_log_pou_denom_ref_ns
    # q_i^*(x) = pi(x) H_i(x) / Hbar(x), so
    #   -∇² log q_i^* = P_post + P_window + ∇² log Hbar.
    P_pou_raw_ns = P_raw_ns + P_window_ref_ns + hess_log_pou_denom_ref_ns
    P_pou_raw_ns = 0.5 * (P_pou_raw_ns + P_pou_raw_ns.transpose(-1, -2))

    eigvals_pou_ns, eigvecs_pou_ns = torch.linalg.eigh(P_pou_raw_ns)
    trusted_pou_ns = (eigvals_pou_ns >= HESS_MIN) & (eigvals_pou_ns <= HESS_MAX)
    prec_eig_pou_ns = torch.where(trusted_pou_ns, eigvals_pou_ns, torch.zeros_like(eigvals_pou_ns))
    P_pou_ref_ns = torch.einsum('nij,nj,nkj->nik', eigvecs_pou_ns, prec_eig_pou_ns, eigvecs_pou_ns) + 1e-6 * I

    s0_pou_eig = torch.einsum('nij,nj->ni', eigvecs_pou_ns.transpose(-1, -2), s0_pou_ns)
    delta_pou_eig = torch.where(
        trusted_pou_ns,
        s0_pou_eig / torch.clamp(prec_eig_pou_ns, min=HESS_MIN),
        torch.zeros_like(s0_pou_eig),
    )
    delta_pou_ns = torch.einsum('nij,nj->ni', eigvecs_pou_ns, delta_pou_eig)
    mu_pou_ref_ns = X_ref_ns + delta_pou_ns

    gated_info_pou_ns = {
        'eigvecs': eigvecs_pou_ns,
        'eigvals': prec_eig_pou_ns,
        'trusted': trusted_pou_ns,
    }

    quad_gain_pou_ns = 0.5 * torch.sum(s0_pou_ns * delta_pou_ns, dim=1)
    eig_for_logdet_pou_ns = torch.where(trusted_pou_ns, eigvals_pou_ns, torch.ones_like(eigvals_pou_ns))
    logdet_P_pou_ns = torch.sum(torch.log(torch.clamp(eig_for_logdet_pou_ns, min=1e-30)), dim=1)
    log_pou_ns = (log_post_x + log_window_norm_ns - log_pou_denom_ref_ns + quad_gain_pou_ns) - 0.5 * logdet_P_pou_ns
    log_window_overlap_ns = log_pou_denom_ref_ns

    norm_prior = torch.norm(s0_prior_ns, dim=1).mean().item()
    norm_lik = torch.norm(grad_lik_ns, dim=1).mean().item()
    avg_ll = log_lik_ns.mean().item()

    print(f"  > Avg Prior Score Norm:       {norm_prior:.4f}")
    print(f"  > Avg Likelihood Grad Norm:   {norm_lik:.4f}")
    print(f"  > Avg Log-Likelihood Value:   {avg_ll:.4f}")

print(f"Precomputation: {time.time()-t0:.2f}s")

_HLSI_CACHE = {}


def _compute_sigmainv_gated(gated_info, et2, var_t):
    eigvecs = gated_info['eigvecs']
    eigvals = gated_info['eigvals']
    trusted = gated_info['trusted']

    hlsi_eig = eigvals / (et2 + var_t * eigvals + 1e-30)
    twd_eig = 1.0 / var_t
    sigmainv_eig = torch.where(trusted, hlsi_eig, twd_eig)
    SigmaInv = torch.einsum('nij,nj,nkj->nik', eigvecs, sigmainv_eig, eigvecs)
    return SigmaInv


def _get_hlsi_time_mats(t, P_ref, mu_ref, gated_info=None):
    t_key = float(t)
    if t_key < 1e-4:
        t_key = 1e-4

    bank_key = (
        t_key,
        int(P_ref.data_ptr()),
        int(mu_ref.data_ptr()),
        int(gated_info['eigvals'].data_ptr()) if gated_info is not None else -1,
    )
    if bank_key in _HLSI_CACHE:
        return _HLSI_CACHE[bank_key]

    et = math.exp(-t_key)
    et2 = et * et
    var_t = 1.0 - math.exp(-2.0 * t_key)

    if gated_info is not None:
        SigmaInv = _compute_sigmainv_gated(gated_info, et2, var_t)
    else:
        N, d = mu_ref.shape
        I = torch.eye(d, device=P_ref.device, dtype=P_ref.dtype).unsqueeze(0)
        A = et2 * I + var_t * P_ref
        SigmaInv = torch.linalg.solve(A, P_ref)

    mu_t = et * mu_ref
    b = torch.einsum('nij,nj->ni', SigmaInv, mu_t)

    _HLSI_CACHE[bank_key] = (SigmaInv, b)
    return SigmaInv, b


def eval_hlsi_posterior_score(y, t, X_ref, log_lik_ref, P_ref, mu_ref, gated_info=None):
    if t < 1e-4:
        t = 1e-4
    w = get_posterior_snis_weights(y, t, X_ref, log_lik_ref)

    SigmaInv_ref, b_ref = _get_hlsi_time_mats(t, P_ref, mu_ref, gated_info=gated_info)
    Abar = torch.einsum('mn,nij->mij', w, SigmaInv_ref)
    term1 = torch.einsum('mij,mj->mi', Abar, y)
    term2 = torch.einsum('mn,nd->md', w, b_ref)
    return -term1 + term2


POU_QUERY_BLOCK = 128
POU_REF_BLOCK = 512


def _compute_log_pou_overlap_penalty(X_ref, mu_ref, eigvecs, prec_eig, trusted,
                                     label='base', query_block=POU_QUERY_BLOCK,
                                     ref_block=POU_REF_BLOCK):
    """
    Practical PoU correction for the WC masses:

        log m_i^{PoU} ≈ log mass_i^WC - log sum_j H(mu_i; j),

    where H(.; j) is the local Gaussian window centered at x_j with the same
    band-gated precision used by the HLSI geometry. We only change the global
    mixing weights here; the particle-wise score signal remains whatever the
    sampler config selects (HLSI, CE-HLSI, etc.).
    """
    n_ref = X_ref.shape[0]
    work_device = X_ref.device
    work_dtype = X_ref.dtype

    eig_for_logdet = torch.where(
        trusted, torch.clamp(prec_eig, min=1e-30), torch.ones_like(prec_eig)
    )
    log_window_norm = 0.5 * torch.sum(torch.log(eig_for_logdet), dim=1)

    print(f"  [{label}] Computing PoU overlap penalties with query_block={query_block}, ref_block={ref_block}...")
    t0_pou = time.time()

    log_overlap = torch.empty((n_ref,), device=work_device, dtype=work_dtype)
    n_query_blocks = (n_ref + query_block - 1) // query_block

    for qb, q0 in enumerate(range(0, n_ref, query_block)):
        q1 = min(q0 + query_block, n_ref)
        mu_block = mu_ref[q0:q1]
        block_max = torch.full((q1 - q0,), -float('inf'), device=work_device, dtype=work_dtype)
        block_sumexp = torch.zeros((q1 - q0,), device=work_device, dtype=work_dtype)

        for r0 in range(0, n_ref, ref_block):
            r1 = min(r0 + ref_block, n_ref)
            X_block = X_ref[r0:r1]
            V_block = eigvecs[r0:r1]
            lam_block = prec_eig[r0:r1]
            norm_block = log_window_norm[r0:r1]

            diff = mu_block.unsqueeze(1) - X_block.unsqueeze(0)
            proj = torch.einsum('qrd,rdk->qrk', diff, V_block)
            mahal = torch.sum((proj ** 2) * lam_block.unsqueeze(0), dim=2)
            log_H = norm_block.unsqueeze(0) - 0.5 * mahal

            local_max = torch.max(log_H, dim=1).values
            new_max = torch.maximum(block_max, local_max)
            block_sumexp = (
                block_sumexp * torch.exp(block_max - new_max)
                + torch.sum(torch.exp(log_H - new_max.unsqueeze(1)), dim=1)
            )
            block_max = new_max

            del X_block, V_block, lam_block, norm_block, diff, proj, mahal, log_H, local_max, new_max

        log_overlap[q0:q1] = block_max + torch.log(torch.clamp(block_sumexp, min=1e-300))

        if (qb + 1) % max(1, n_query_blocks // 10) == 0 or q1 == n_ref:
            print(f"    [{label}] PoU overlap block {qb + 1}/{n_query_blocks} complete")

        del mu_block, block_max, block_sumexp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"  [{label}] PoU overlap time: {time.time() - t0_pou:.2f}s")
    return log_overlap


PRECOMP_NS = {
    'X_ref': X_ref_ns,
    's0_post_ref': s0_post_ns,
    's0_pou_ref': s0_pou_ns,
    'log_lik_ref': log_lik_ns,
    'log_mass_ref': log_mass_ns,
    'log_pou_ref': log_pou_ns,
    'log_window_overlap_ref': log_window_overlap_ns,
    'log_pou_denom_ref': log_pou_denom_ref_ns,
    'grad_log_pou_denom_ref': grad_log_pou_denom_ref_ns,
    'hess_log_pou_denom_ref': hess_log_pou_denom_ref_ns,
    'P_ref': P_ref_ns,
    'mu_ref': mu_ref_ns,
    'gated_info': gated_info_ns,
    'P_pou_ref': P_pou_ref_ns,
    'mu_pou_ref': mu_pou_ref_ns,
    'gated_info_pou': gated_info_pou_ns,
}

DEFAULT_N_GEN_NS = 500

SAMPLER_CONFIGS = OrderedDict([
    (
        'MALA (prior)',
        {
            'init': 'prior',
            'init_steps': 0,
            'mala_steps': 400,
            'mala_burnin': 100,
            'mala_dt': 1e-4,
            'is_reference': True,
        },
    ),

     (
        'Tweedie',
        {
            'init': 'tweedie',
            'init_weights': 'L',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),

    (
        'Blend',
        {
            'init': 'blend',
            'init_weights': 'L',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': False,
        },
    ),


    (
        'HLSI',
        {
            'init': 'HLSI',
            'init_weights': 'L',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
    (
        'WC-HLSI',
        {
            'init': 'HLSI',
            'init_weights': 'WC',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
    (
        'PoU-HLSI',
        {
            'init': 'HLSI',
            'init_weights': 'PoU',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
    (
        'CE-HLSI',
        {
            'init': 'CE-HLSI',
            'init_weights': 'L',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
    (
        'CE-WC-HLSI',
        {
            'init': 'CE-HLSI',
            'init_weights': 'WC',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
   (
        'CE-PoU-HLSI',
        {
            'init': 'CE-HLSI',
            'init_weights': 'PoU',
            'init_steps': 200,
            'mala_steps': 0,
            'mala_burnin': 0,
            'log_mean_ess': True,
        },
    ),
])



def run_single_sampler_config(label, config, prior_model, lik_model, precomp):
    cfg = normalize_sampler_config(label, config, DEFAULT_N_GEN_NS, ACTIVE_DIM)
    display_name = cfg['display_name']

    print(f"\n=== Running {display_name} ===")
    print(
        f"  init={cfg['init']} | init_weights={cfg['init_weights']} | "
        f"init_steps={cfg['init_steps']} | mala_steps={cfg['mala_steps']} | "
        f"mala_burnin={cfg['mala_burnin']} | mala_dt={cfg['mala_dt']}"
    )

    init_samples = None
    ess_trace = None

    if cfg['init'] != 'prior' and cfg['init_steps'] > 0:
        bank = get_sampler_precomp_bank(precomp, cfg['init'], cfg['init_weights'])
        init_log_weights = get_sampler_log_weights(cfg['init_weights'], bank)
        init_out = run_sampler_heun(
            cfg['n_samples'], cfg['init'],
            bank['X_ref'], bank['s0_post_ref'], init_log_weights,
            P_ref=bank['P_ref'], mu_ref=bank['mu_ref'],
            gated_info=bank['gated_info'],
            steps=cfg['init_steps'], dim=cfg['dim'],
            log_mean_ess=cfg['log_mean_ess'],
            t_max=cfg['init_tmax'], t_min=cfg['init_tmin'],
        )
        if cfg['log_mean_ess']:
            init_samples, ess_trace = init_out
        else:
            init_samples = init_out
    else:
        init_samples = prior_model.sample(cfg['n_samples'])

    mala_info = None
    final_samples = init_samples
    if cfg['mala_steps'] > 0:
        final_samples, mala_info = run_mala_sampler(
            cfg['n_samples'], prior_model, lik_model,
            steps=cfg['mala_steps'], dt=cfg['mala_dt'], burn_in=cfg['mala_burnin'],
            x_init=init_samples, verbose=True, return_info=True,
        )

    run_info = dict(cfg)
    run_info['init_bank'] = 'PoU' if (cfg['init'] != 'prior' and canonicalize_init_weights(cfg['init_weights']) == 'PoU') else 'base'
    run_info['init_log_weights'] = get_sampler_log_weight_name(cfg['init_weights']) if cfg['init'] != 'prior' else 'prior'
    if mala_info is not None:
        run_info.update(mala_info)

    return final_samples.cpu(), ess_trace, run_info


def run_sampler_suite(sampler_configs, prior_model, lik_model, precomp):
    samples = OrderedDict()
    ess_logs = OrderedDict()
    run_info = OrderedDict()

    for label, cfg in sampler_configs.items():
        t_start = time.time()
        samps, ess_trace, info = run_single_sampler_config(label, cfg, prior_model, lik_model, precomp)
        runtime_seconds = time.time() - t_start
        samples[label] = samps
        if ess_trace is not None and len(ess_trace.get('t', [])) > 0:
            ess_logs[label] = ess_trace
        info = dict(info)
        info['runtime_seconds'] = float(runtime_seconds)
        run_info[label] = info
        print(f"{label}: {runtime_seconds:.2f}s")

    return samples, ess_logs, run_info


samples_ns, ess_logs_ns, sampler_run_info_ns = run_sampler_suite(
    SAMPLER_CONFIGS_NS,
    prior_model_ns,
    lik_model_ns,
    PRECOMP_NS,
)

display_names_ns = {label: info.get('display_name', label) for label, info in sampler_run_info_ns.items()}
reference_key_ns = choose_reference_key(samples_ns, sampler_run_info_ns)
reference_title_ns = display_names_ns.get(reference_key_ns, reference_key_ns)

print('\n=== Mean ESS vs t (across particles) ===')
plot_mean_ess_logs(ess_logs_ns, display_names=display_names_ns)

print('\n=== Evaluation (latent/coordinate metrics) ===')
print(
    f"{'Method':<24} | {'RMSE_alpha':<10} | {'RelL2_alpha':<11} | "
    f"{('MMD->' + reference_title_ns)[:14]:<14} | {'KSD':<10} | {'KLdiag':<10}"
)
print('-' * 95)

ref_clean = robust_clean_samples(samples_ns[reference_key_ns])
alpha_true_t = torch.tensor(alpha_true_np, device=device, dtype=torch.float64)
metrics_ns = {}

for label, raw in samples_ns.items():
    samps = robust_clean_samples(raw)
    if samps.shape[0] < 50:
        continue

    mean_latent = torch.mean(samps, dim=0)
    rmse_alpha = rmse_vec(mean_latent, alpha_true_t)
    rel_alpha = rel_l2_vec(mean_latent, alpha_true_t)
    mmd = compute_mmd_rbf(samps, ref_clean)
    ksd = compute_multiscale_ksd(samps, posterior_score_fn)
    kl = compute_kl_divergence(samps, prior_model_ns, lik_model_ns)

    metrics_ns[label] = dict(
        mean_latent=mean_latent.detach().cpu().numpy(),
        RMSE_alpha=rmse_alpha,
        RelL2_alpha=rel_alpha,
        MMD_to_reference=mmd,
        KSD=ksd,
        KLdiag=kl,
    )

    print(
        f"{display_names_ns.get(label, label):<24} | {rmse_alpha:<10.4f} | {rel_alpha:<11.4f} | "
        f"{mmd:<14.4f} | {ksd:<10.4f} | {kl:<10.4f}"
    )

print('\n=== PCA Histograms ===')
plot_pca_histograms(
    samples_ns,
    alpha_true_np,
    display_names=display_names_ns,
    reference_key=reference_key_ns,
)

# Cell 7: Scientific Visualization & Physics Metrics (Navier-Stokes)
import matplotlib.pyplot as plt
import numpy as np
import torch
import jax.numpy as jnp

if 'metrics_ns' not in globals() or metrics_ns is None:
    metrics_ns = {}

try:
    Basis_np = np.array(Basis)
except NameError:
    print("Warning: 'Basis' matrix not found. Ensure Cell 1 is run.")
    Basis_np = np.random.randn(32 * 32, 24)

y_clean_np = np.array(y_clean)


def reconstruct_vorticity(latents):
    if isinstance(latents, torch.Tensor):
        latents = latents.detach().cpu().numpy()

    latents = np.asarray(latents)
    if latents.ndim == 1:
        latents = latents[None, :]

    d_lat = latents.shape[1]
    if Basis_np.shape[1] < d_lat:
        raise ValueError(f"Basis has only {Basis_np.shape[1]} columns but latents have dim {d_lat}.")

    B = Basis_np[:, :d_lat]
    fields_flat = latents @ B.T
    N = latents.shape[0]
    grid_size = int(np.sqrt(fields_flat.shape[1]))
    return fields_flat.reshape(N, grid_size, grid_size)


def get_valid_samples(samps):
    if isinstance(samps, torch.Tensor):
        samps = samps.detach().cpu().numpy()
    samps = np.asarray(samps)
    return samps[np.isfinite(samps).all(axis=1)]


def rmse_array(a_hat, a_true):
    a_hat = np.asarray(a_hat)
    a_true = np.asarray(a_true)
    return float(np.sqrt(np.mean((a_hat - a_true) ** 2)))


if 'ACTIVE_DIM' in globals():
    d_lat = int(ACTIVE_DIM)
else:
    any_key = next(iter(samples_ns.keys()))
    d_lat = int(get_valid_samples(samples_ns[any_key]).shape[1])

alpha_true_lat = np.asarray(alpha_true_np).reshape(-1)[:d_lat]
true_field = reconstruct_vorticity(alpha_true_lat)[0]
norm_true = np.linalg.norm(true_field) + 1e-12
norm_y_clean = np.linalg.norm(y_clean_np) + 1e-12

print('\n=== Physical Parameter Space Metrics (Vorticity + Forward) ===')
print(f"{'Method':<24} | {'Inv RelL2(%)':<12} | {'RMSE_alpha':<12} | {'FwdRelErr':<12}")
print('-' * 76)

mean_fields = {}
for label, samps in samples_ns.items():
    samps_clean = get_valid_samples(samps)
    if samps_clean.shape[0] < 10:
        print(f"{display_names_ns.get(label, label):<24} | FAILED (Unstable / too few valid)")
        continue

    mean_latent = np.mean(samps_clean, axis=0)[:d_lat]
    mean_field = reconstruct_vorticity(mean_latent)[0]
    mean_fields[label] = mean_field

    inv_rel_l2_pct = 100.0 * (np.linalg.norm(mean_field - true_field) / norm_true)
    rmse_alpha = rmse_array(mean_latent, alpha_true_lat)

    try:
        y_pred = np.array(solve_forward(jnp.array(mean_latent)))
        fwd_rel = float(np.linalg.norm(y_pred - y_clean_np) / norm_y_clean)
    except Exception:
        fwd_rel = float('nan')

    if label not in metrics_ns:
        metrics_ns[label] = {}
    metrics_ns[label].update(dict(
        mean_latent=mean_latent,
        RMSE_alpha=rmse_alpha,
        RMSE_field=rmse_array(mean_field, true_field),
        RelL2_field=float(np.linalg.norm(mean_field - true_field) / norm_true),
        FwdRelErr=fwd_rel,
    ))

    print(f"{display_names_ns.get(label, label):<24} | {inv_rel_l2_pct:<12.4f} | {rmse_alpha:<12.4e} | {fwd_rel:<12.4e}")

print('\n=== Final NS metrics (paper-ready) ===')
print(
    f"{'Method':<24} | {'RMSE_a':<9} | {'RMSE_w':<9} | {'FwdRel':<9} | "
    f"{('MMD->' + reference_title_ns)[:12]:<12} | {'KSD':<9} | {'KLdiag':<9}"
)
print('-' * 98)
for label, data in metrics_ns.items():
    mmd = data.get('MMD_to_reference', float('nan'))
    ksd = data.get('KSD', float('nan'))
    kld = data.get('KLdiag', float('nan'))
    print(
        f"{display_names_ns.get(label, label):<24} | {data.get('RMSE_alpha', float('nan')):<9.3e} | "
        f"{data.get('RMSE_field', float('nan')):<9.3e} | {data.get('FwdRelErr', float('nan')):<9.3e} | "
        f"{mmd:<12.3e} | {ksd:<9.3e} | {kld:<9.3e}"
    )

# ==========================================
# RESULTS DATAFRAME + RUN LOG SAVING
# ==========================================

def _results_method_family(label, info):
    raw_init_mode = info.get('init', label)
    label_l = str(label).lower()
    try:
        init_mode = canonicalize_init_name(raw_init_mode) if str(raw_init_mode).lower() != 'prior' else 'prior'
    except Exception:
        init_mode = str(raw_init_mode)
    if 'mala' in label_l or init_mode == 'prior':
        return 'Prior'
    family_map = {
        'tweedie': 'Tweedie',
        'blend_posterior': 'Blend',
        'hlsi_posterior': 'HLSI',
        'ce_hlsi': 'CE-HLSI',
    }
    return family_map.get(init_mode, str(raw_init_mode))


def _results_weight_mode(label, info):
    raw_init_mode = info.get('init', label)
    if 'mala' in str(label).lower() or str(raw_init_mode).lower() == 'prior':
        return 'prior'
    try:
        return canonicalize_init_weights(info.get('init_weights', 'L'))
    except Exception:
        return str(info.get('init_weights', 'L'))


def build_results_dataframes(metrics_dict, run_info_dict, n_ref, target_name, reference_name=None):
    metric_rows = [
        'RMSE_alpha',
        'RelL2_alpha',
        'MMD_to_reference',
        'KSD',
        'KLdiag',
        'RMSE_field',
        'RelL2_field',
        'FwdRelErr',
    ]

    ordered_methods = [label for label in run_info_dict.keys() if label in metrics_dict]
    results_df = pd.DataFrame(index=metric_rows, columns=ordered_methods, dtype=np.float64)
    results_df.index.name = 'metric'

    runinfo_rows = []
    for label in ordered_methods:
        info = dict(run_info_dict[label])
        metric_dict = metrics_dict.get(label, {})
        for metric_name in metric_rows:
            results_df.loc[metric_name, label] = metric_dict.get(metric_name, np.nan)

        runinfo_rows.append({
            'target': target_name,
            'label': label,
            'display_name': display_names_ns.get(label, label),
            'method': _results_method_family(label, info),
            'weight_mode': _results_weight_mode(label, info),
            'N_ref': int(n_ref),
            'steps': int(info.get('init_steps', 0)),
            'mala_steps': int(info.get('mala_steps', info.get('steps', 0))),
            'mala_burnin': int(info.get('mala_burnin', info.get('burn_in', 0))),
            'mala_step_size': float(info.get('mala_dt', info.get('dt', np.nan))),
            'runtime_seconds': float(info.get('runtime_seconds', np.nan)),
            'reference_method': reference_name,
        })

    results_runinfo_df = pd.DataFrame(runinfo_rows)
    return results_df, results_runinfo_df


RESULTS_DIR = RUN_RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_STEM = RUN_RESULTS_STEM

results_df, results_runinfo_df = build_results_dataframes(
    metrics_ns,
    sampler_run_info_ns,
    n_ref=N_REF_NS,
    target_name='Navier-Stokes inversion',
    reference_name=reference_title_ns,
)

results_df_path = os.path.join(RESULTS_DIR, f'{RESULTS_STEM}_metrics.csv')
results_runinfo_df_path = os.path.join(RESULTS_DIR, f'{RESULTS_STEM}_runinfo.csv')
results_df.to_csv(results_df_path)
results_runinfo_df.to_csv(results_runinfo_df_path, index=False)

print(f'\nSaved results dataframe to {results_df_path}')
print(f'Saved run-info dataframe to {results_runinfo_df_path}')

save_reproducibility_log(extra_sections={
    'External basis-mode source': {
        'required_local_file': 'data/Basis_Modes.csv',
        'github_reference': 'https://github.com/nguyenvanhaibk92/mcTagent_digital_twin/tree/main/NS_eq/data',
        'note': 'The Navier-Stokes basis-mode CSV is an external input dependency and is not fully reconstructible from this script alone.',
    },
    'Results files': {
        'metrics_csv': results_df_path,
        'runinfo_csv': results_runinfo_df_path,
    },
})

print('\nVisualizing Vorticity Fields and Error Maps...')
methods_to_plot = [label for label in samples_ns.keys() if (label in mean_fields)]

n_cols = len(methods_to_plot) + 1
fig, axes = plt.subplots(4, n_cols, figsize=(4 * n_cols, 14))

# Use the ground-truth vorticity field to set the color normalization for the
# field visualizations (top and bottom rows). This avoids the display scale
# being distorted when a sampler like MALA/reference collapses or blows up.
vmin = float(np.min(true_field))
vmax = float(np.max(true_field))

vis_anchor_key = reference_key_ns if reference_key_ns in mean_fields else next(iter(mean_fields.keys()))
if vis_anchor_key in samples_ns and vis_anchor_key in mean_fields:
    anchor_vis_samps = get_valid_samples(samples_ns[vis_anchor_key])[:1000]
    if anchor_vis_samps.shape[0] > 0:
        anchor_vis_fields = reconstruct_vorticity(anchor_vis_samps[:, :d_lat])
        max_err = max(1e-12, float(np.abs(mean_fields[vis_anchor_key] - true_field).max()))
        max_std = max(1e-12, float(np.std(anchor_vis_fields, axis=0).max()))
    else:
        max_err = 1e-12
        max_std = 1e-12
else:
    max_err = 1e-12
    max_std = 1e-12

im0 = axes[0, 0].imshow(true_field, cmap='RdBu_r', origin='lower', vmin=vmin, vmax=vmax)
axes[0, 0].set_title('Ground Truth\nInitial Vorticity $\\omega_0$', fontsize=18)
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

axes[3, 0].imshow(true_field, cmap='RdBu_r', origin='lower', vmin=vmin, vmax=vmax)
axes[3, 0].set_title('Ground Truth', fontsize=14)
axes[3, 0].axis('off')
axes[1, 0].axis('off')
axes[2, 0].axis('off')

if vis_anchor_key not in mean_fields:
    max_err = 1e-12
    max_std = 1e-12
    for label in methods_to_plot:
        mean_f = mean_fields[label]
        max_err = max(max_err, np.abs(mean_f - true_field).max())
        samps = get_valid_samples(samples_ns[label])[:500]
        if samps.shape[0] > 0:
            fields = reconstruct_vorticity(samps[:, :d_lat])
            max_std = max(max_std, np.std(fields, axis=0).max())

for i, label in enumerate(methods_to_plot):
    col = i + 1
    mean_f = mean_fields[label]

    axes[0, col].imshow(mean_f, cmap='RdBu_r', origin='lower', vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f"{display_names_ns.get(label, label)}\nMean Posterior", fontsize=18)
    axes[0, col].axis('off')

    err_f = np.abs(mean_f - true_field)
    axes[1, col].imshow(err_f, cmap='inferno', origin='lower', vmin=0, vmax=max_err)
    axes[1, col].set_title(f"Error Map\n(Max: {err_f.max():.2f})", fontsize=16)
    axes[1, col].axis('off')

    samps = get_valid_samples(samples_ns[label])[:1000]
    if samps.shape[0] > 0:
        fields = reconstruct_vorticity(samps[:, :d_lat])
        std_f = np.std(fields, axis=0)
    else:
        fields = None
        std_f = np.zeros_like(true_field)

    im_std = axes[2, col].imshow(std_f, cmap='viridis', origin='lower', vmin=0, vmax=max_std)
    axes[2, col].set_title('Uncertainty $\\sigma$', fontsize=16)
    axes[2, col].axis('off')
    plt.colorbar(im_std, ax=axes[2, col], fraction=0.046, pad=0.04)

    if fields is not None and samps.shape[0] > 0:
        samp_f = fields[-1]
        im_samp = axes[3, col].imshow(samp_f, cmap='RdBu_r', origin='lower', vmin=vmin, vmax=vmax)
        axes[3, col].set_title('Random Sample', fontsize=14)
        axes[3, col].axis('off')
        plt.colorbar(im_samp, ax=axes[3, col], fraction=0.046, pad=0.04)
    else:
        axes[3, col].axis('off')

plt.suptitle(f"Inverse Navier-Stokes (Latent Dim={d_lat}): Vorticity Reconstruction", fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

run_results_zip_path = zip_run_results_dir()
print(f'Run-results directory: {RUN_RESULTS_DIR}')
print(f'Run-results zip: {run_results_zip_path}')
print("\n=== Navier-Stokes inversion HLSI pipeline complete ===")